# 23. Regime Effect Summary — Facial Skincare

This notebook synthesizes the regime-level stage-allocation evidence for LightGBM and the Transformer at unconditional NDCG@5 and candidate depth 1,000. It reports overall, cold, weak, moderate, strong, and pooled non-cold results for five fixed contrasts.

Condition means come from Notebook 15, while paired case deltas and intervals come from the archived Notebook 16 outputs. The notebook does not reconstruct ranks or canonical case metrics. It independently estimates Strong-minus-Weak differences by resampling the two disjoint regime groups, using 10,000 bootstrap replicates.

All regime results are exploratory and unadjusted. The evidence scorecard localizes observed patterns but does not select a model, assign a mechanism, or extend the thesis’s confirmatory family.

## Scope Summary

The report covers all five fixed contrasts for LightGBM and the Transformer across overall, cold, weak, moderate, strong, and non-cold scopes. Strong-minus-Weak differences compare independent user groups and are never treated as paired case contrasts.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import re

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


Mounted at /content/drive


## Context and Contrasts

The five reported contrasts are:

1. S1-P − S1-Q: Stage 1 personalization on NDCG@5, secondary to the Stage 1 Recall@1,000 endpoint.
2. Base − S1-Q: no-prior reranking benefit.
3. RankP − Base: prior-feature contribution on the fixed query-only pool.
4. Full − RankP: pool-policy contrast given prior-aware reranking.
5. Full − Base: combined joint-policy difference.

The Stage 1 contrast is shared across rerankers and is repeated only for presentation. All Stage 2 contrasts retain the same reranker family on both sides. Full − RankP changes the candidate pool and own-pool fit together and therefore does not isolate candidate-source variation alone.

In [20]:
# ==== Fixed Contract — Declared Before Reading Performance Files ====
NOTEBOOK_NAME = "26_regime_effect_summary_face.ipynb"
CATEGORY_ID = "face"
CATEGORY_KEY = "facial_skincare"
CATEGORY_LABEL = "Facial Skincare"
REVISION_DATE = "2026-07-21"

DEFAULT_PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/facial_skincare")
PROJECT_ROOT = Path(os.environ.get("THESIS_CATEGORY_ROOT", DEFAULT_PROJECT_ROOT))
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

NOTEBOOK15_DIR = OUTPUTS_DIR / "analysis" / "stage_allocation_summary"
NOTEBOOK15_OVERALL_PATH = NOTEBOOK15_DIR / "stage_allocation_overall_by_reranker.csv"
NOTEBOOK15_BY_REGIME_PATH = NOTEBOOK15_DIR / "stage_allocation_by_regime_pool_depth.csv"
NOTEBOOK15_MANIFEST_PATH = NOTEBOOK15_DIR / "stage_allocation_manifest.json"

NOTEBOOK16_DIR = OUTPUTS_DIR / "analysis" / "paired_significance_summary"
NOTEBOOK16_DELTAS_PATH = NOTEBOOK16_DIR / "paired_case_deltas_report_depth.parquet"
NOTEBOOK16_OVERALL_PATH = NOTEBOOK16_DIR / "paired_inference_summary_report_depth.csv"
NOTEBOOK16_BY_REGIME_PATH = NOTEBOOK16_DIR / "paired_inference_by_regime_report_depth.csv"
NOTEBOOK16_COVERAGE_PATH = NOTEBOOK16_DIR / "pair_coverage_qc.csv"
NOTEBOOK16_MANIFEST_PATH = NOTEBOOK16_DIR / "run_manifest.json"

NOTEBOOK14_MANIFEST_PATH = OUTPUTS_DIR / "pipeline_aggregate" / "pipeline_manifest.json"

OUT_DIR = OUTPUTS_DIR / "analysis" / "regime_analysis_report"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILES = {
    "regime_effect_summary_report_depth": OUT_DIR / "regime_effect_summary_report_depth.csv",
    "strong_weak_difference_in_delta": OUT_DIR / "strong_weak_difference_in_delta.csv",
    "regime_evidence_scorecard": OUT_DIR / "regime_evidence_scorecard.csv",
    "regime_pair_coverage_qc": OUT_DIR / "regime_pair_coverage_qc.csv",
    "run_manifest": OUT_DIR / "run_manifest.json",
}

PRIMARY_METRIC_NAME = "NDCG"
PRIMARY_METRIC_CUTOFF = 5
REPORT_POOL_DEPTH = 1000  # report depth 1000 (2026-07-23); reranking always top-1000; all-depth panels unchanged
RERANKER_FAMILIES = ["lightgbm", "transformer"]
RERANKER_LABELS = {"lightgbm": "LightGBM", "transformer": "Transformer"}
REGIME_SCOPES = ["overall", "cold", "weak", "moderate", "strong", "non-cold"]
CORE_REGIMES = ["cold", "weak", "moderate", "strong"]
REGIME_ORDER = {value: index for index, value in enumerate(REGIME_SCOPES)}
STRONG_WEAK_BOOTSTRAP_ITERATIONS = 10_000
STRONG_WEAK_BOOTSTRAP_BASE_SEED = 20260729
SIMULATION_BATCH_SIZE = 256
DELTA_TIE_ATOL = 1e-12

COMPARISON_DEFINITIONS = [
    {
        "comparison_order": 1,
        "comparison_id": "P1_only_minus_P0",
        "comparison_label": "P1-only − P0",
        "effect_label": "Stage 1 personalization effect",
        "left_condition": "P1-only",
        "right_condition": "P0",
        "source_suffix": "P1_only_minus_P0",
        "shared_retrieval_effect": True,
    },
    {
        "comparison_order": 2,
        "comparison_id": "P2_Q_minus_P0",
        "comparison_label": "P2-Q − P0",
        "effect_label": "No-prior reranking effect",
        "left_condition": "P2-Q",
        "right_condition": "P0",
        "source_suffix": "P2_Q_minus_P0",
        "shared_retrieval_effect": False,
    },
    {
        "comparison_order": 3,
        "comparison_id": "P2_P_minus_P2_Q",
        "comparison_label": "P2-P − P2-Q",
        "effect_label": "Stage 2 prior-feature effect",
        "left_condition": "P2-P",
        "right_condition": "P2-Q",
        "source_suffix": "P2_P_minus_P2_Q",
        "shared_retrieval_effect": False,
    },
    {
        "comparison_order": 4,
        "comparison_id": "Full_minus_P2_P",
        "comparison_label": "Full − P2-P",
        "effect_label": "Personalized candidate-source effect",
        "left_condition": "Full",
        "right_condition": "P2-P",
        "source_suffix": "Full_minus_P2_P",
        "shared_retrieval_effect": False,
    },
    {
        "comparison_order": 5,
        "comparison_id": "Full_minus_P2_Q",
        "comparison_label": "Full − P2-Q",
        "effect_label": "Combined personalization effect",
        "left_condition": "Full",
        "right_condition": "P2-Q",
        "source_suffix": "Full_minus_P2_Q",
        "shared_retrieval_effect": False,
    },
]
print("Category:", CATEGORY_LABEL)
print("Fixed contract: NDCG@5 at candidate pool depth 1000")
print("Rerankers:", RERANKER_FAMILIES)


Category: Facial Skincare
Fixed contract: NDCG@5 at candidate pool depth 1000
Rerankers: ['lightgbm', 'transformer']


In [21]:
# ==== Shared Validation and Resampling Helpers ====
def require_columns(frame, required_columns, label):
    missing = sorted(set(required_columns).difference(frame.columns))
    if missing:
        raise RuntimeError(f"{label} is missing required columns: {missing}")


def require_unique_columns(frame, label):
    duplicated = frame.columns[frame.columns.duplicated()].astype(str).tolist()
    if duplicated:
        raise RuntimeError(f"{label} has duplicated columns: {duplicated}")


def normalize_text(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_lower(series):
    return series.fillna("").astype(str).map(normalize_text).str.lower()


def parse_bool_series(series, label):
    if pd.api.types.is_bool_dtype(series):
        parsed = series.astype("boolean")
    else:
        text = normalize_lower(series)
        numeric = pd.to_numeric(series, errors="coerce")
        mapped = text.map({
            "true": True, "false": False,
            "yes": True, "no": False,
            "1": True, "0": False,
        })
        parsed = mapped.where(
            mapped.notna(), numeric.map({1.0: True, 0.0: False})
        ).astype("boolean")
    if parsed.isna().any():
        raise RuntimeError(f"{label} contains missing or non-boolean values.")
    return parsed.astype(bool)


def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def make_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): make_jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [make_jsonable(item) for item in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if pd.isna(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    if value is pd.NA or (isinstance(value, float) and math.isnan(value)):
        return None
    return value


def write_json(path, payload):
    Path(path).write_text(
        json.dumps(make_jsonable(payload), ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def stable_seed(*parts):
    payload = "|".join(map(str, (STRONG_WEAK_BOOTSTRAP_BASE_SEED, *parts)))
    digest = hashlib.sha256(payload.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little") % (2**32 - 1)


def source_comparison_id(reranker_family, comparison):
    if comparison["shared_retrieval_effect"]:
        return comparison["source_suffix"]
    return f"{reranker_family}_{comparison['source_suffix']}"


def independent_stratified_bootstrap_mean_difference(
    strong_values, weak_values, seed
):
    strong_values = np.asarray(strong_values, dtype=float)
    weak_values = np.asarray(weak_values, dtype=float)
    if (
        strong_values.ndim != 1
        or weak_values.ndim != 1
        or strong_values.size == 0
        or weak_values.size == 0
        or not np.isfinite(strong_values).all()
        or not np.isfinite(weak_values).all()
    ):
        raise RuntimeError(
            "Independent Strong–Weak bootstrap requires finite, non-empty arrays."
        )
    rng = np.random.default_rng(seed)
    simulated_differences = np.empty(
        STRONG_WEAK_BOOTSTRAP_ITERATIONS, dtype=float
    )
    for start in range(
        0, STRONG_WEAK_BOOTSTRAP_ITERATIONS, SIMULATION_BATCH_SIZE
    ):
        stop = min(
            start + SIMULATION_BATCH_SIZE,
            STRONG_WEAK_BOOTSTRAP_ITERATIONS,
        )
        strong_indices = rng.integers(
            0,
            strong_values.size,
            size=(stop - start, strong_values.size),
        )
        weak_indices = rng.integers(
            0,
            weak_values.size,
            size=(stop - start, weak_values.size),
        )
        simulated_differences[start:stop] = (
            strong_values[strong_indices].mean(axis=1)
            - weak_values[weak_indices].mean(axis=1)
        )
    lower, upper = np.quantile(
        simulated_differences, [0.025, 0.975]
    )
    return float(lower), float(upper)


## Data and Inferential Scope

Input roles are:

- Notebook 15: descriptive condition means and case counts.
- Notebook 16: archived paired case deltas, intervals, p-values, and pair coverage.
- Notebook 14 manifest: category provenance and canonical-quality gates only.

No candidate pool, rank list, or Notebook 14 case-metric table is loaded. The archived Notebook 16 eight-test Holm field is not used to define the final thesis confirmatory family. Regime rows remain exploratory and unadjusted, and this notebook creates no new confirmatory test.

In [22]:
# ==== Validate Manifests and cross-notebook Lineage ====
required_input_paths = [
    NOTEBOOK15_OVERALL_PATH,
    NOTEBOOK15_BY_REGIME_PATH,
    NOTEBOOK15_MANIFEST_PATH,
    NOTEBOOK16_DELTAS_PATH,
    NOTEBOOK16_OVERALL_PATH,
    NOTEBOOK16_BY_REGIME_PATH,
    NOTEBOOK16_COVERAGE_PATH,
    NOTEBOOK16_MANIFEST_PATH,
    NOTEBOOK14_MANIFEST_PATH,
]
missing_paths = [str(path) for path in required_input_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing required Notebook 15/16/14 artifacts: {missing_paths}")

notebook14_manifest = load_json(NOTEBOOK14_MANIFEST_PATH)
notebook15_manifest = load_json(NOTEBOOK15_MANIFEST_PATH)
notebook16_manifest = load_json(NOTEBOOK16_MANIFEST_PATH)

for label, manifest in [
    ("Notebook 14", notebook14_manifest),
    ("Notebook 15", notebook15_manifest),
    ("Notebook 16", notebook16_manifest),
]:
    if manifest.get("category_id") != CATEGORY_ID:
        raise RuntimeError(f"{label} manifest category_id mismatch.")

required_notebook14_checks = [
    "all_five_conditions",
    "all_reranker_families",
    "shared_baselines_stored_once",
    "pool_depth_and_metric_cutoff_separate",
    "candidate_counts_within_pool",
    "target_rank_one_based",
    "target_rank_within_candidate_count",
    "target_absent_metrics_zero",
    "no_duplicated_canonical_key",
    "no_model_selection",
    "all_source_metrics_validated",
    "paired_case_universes_match",
    "qchs_fallback_cases_preserved",
    "no_missing_expected_cells",
]
notebook14_validation = notebook14_manifest.get("validation_results", {})
failed_notebook14_checks = [
    key for key in required_notebook14_checks
    if notebook14_validation.get(key) is not True
]
if failed_notebook14_checks:
    raise RuntimeError(
        f"Notebook 14 canonical-quality gates failed: {failed_notebook14_checks}"
    )
pool_contract = notebook14_manifest.get("pool_depth_contract", {})
if REPORT_POOL_DEPTH not in [
    int(value) for value in pool_contract.get("candidate_pool_depths", [])
]:
    raise RuntimeError("Notebook 14 manifest lacks report pool depth 1000.")
if PRIMARY_METRIC_CUTOFF not in [
    int(value) for value in pool_contract.get("metric_cutoffs", [])
]:
    raise RuntimeError("Notebook 14 manifest lacks metric cutoff 5.")
if pool_contract.get("dimensions_are_distinct") is not True:
    raise RuntimeError("Notebook 14 did not separate pool depth and cutoff.")

notebook15_validation = notebook15_manifest.get("validation_results", {})
required_notebook15_checks = [
    "case_query_user_grain_valid",
    "canonical_ndcg_at_5_matches_target_rank",
    "notebook14_summary_matches_canonical_raw",
    "required_primary_contrasts_present",
    "required_primary_populations_present",
    "output_columns_unique",
]
failed_notebook15_checks = [
    key for key in required_notebook15_checks
    if notebook15_validation.get(key) is not True
]
if failed_notebook15_checks:
    raise RuntimeError(
        f"Notebook 15 descriptive-source gates failed: {failed_notebook15_checks}"
    )
if notebook15_manifest.get("ready_for_thesis_reporting") is not True:
    raise RuntimeError("Notebook 15 is not marked ready for thesis reporting.")
notebook15_contract = notebook15_manifest.get(
    "primary_evaluation_contract", {}
)
if (
    int(notebook15_contract.get("candidate_pool_depth", -1)) != REPORT_POOL_DEPTH
    or notebook15_contract.get("metric_name") != PRIMARY_METRIC_NAME
    or int(notebook15_contract.get("metric_cutoff", -1)) != PRIMARY_METRIC_CUTOFF
):
    raise RuntimeError("Notebook 15 primary evaluation contract mismatch.")

notebook16_validation = notebook16_manifest.get("validation_results", {})
required_notebook16_checks = [
    "notebook14_manifest_required_checks_passed",
    "notebook14_canonical_rank_qc_passed",
    "notebook14_paired_universe_qc_passed",
    "authoritative_inputs_only",
    "metric_fixed_before_outcome_load",
    "pool_depth_fixed_before_outcome_load",
    "one_row_per_case_condition_method_family",
    "one_case_per_user",
    "target_absent_ndcg_zero_preserved",
    "strict_case_id_pairing",
    "regime_consistency_across_pairs",
    "no_unmatched_primary_cases",
    "all_prior_comparisons_method_family_matched",
    "no_method_or_winner_selection",
    "no_rank_reconstruction",
    "finite_simulation_correction_applied",
    "holm_adjustment_applied_to_locked_eight_principal_tests",
    "overall_and_strong_scopes_reported",
    "output_columns_unique",
]
failed_notebook16_checks = [
    key for key in required_notebook16_checks
    if notebook16_validation.get(key) is not True
]
if failed_notebook16_checks:
    raise RuntimeError(
        f"Notebook 16 paired-inference gates failed: {failed_notebook16_checks}"
    )
outcome_filter = notebook16_manifest.get(
    "outcome_row_filter_fixed_before_load", {}
)
if outcome_filter != {
    "metric_name": PRIMARY_METRIC_NAME,
    "metric_cutoff": PRIMARY_METRIC_CUTOFF,
    "candidate_pool_depth": REPORT_POOL_DEPTH,
}:
    raise RuntimeError("Notebook 16 fixed outcome contract mismatch.")

notebook15_outputs = notebook15_manifest.get("output_files", {})
expected_notebook15_outputs = {
    "overall_by_reranker": NOTEBOOK15_OVERALL_PATH,
    "by_regime_pool_depth": NOTEBOOK15_BY_REGIME_PATH,
    "manifest": NOTEBOOK15_MANIFEST_PATH,
}
for key, expected_path in expected_notebook15_outputs.items():
    if Path(notebook15_outputs.get(key, "")) != expected_path:
        raise RuntimeError(f"Notebook 15 output path mismatch: {key}")

notebook16_outputs = notebook16_manifest.get("output_paths", {})
expected_notebook16_outputs = {
    "paired_case_deltas_report_depth": NOTEBOOK16_DELTAS_PATH,
    "paired_inference_summary_report_depth": NOTEBOOK16_OVERALL_PATH,
    "paired_inference_by_regime_report_depth": NOTEBOOK16_BY_REGIME_PATH,
    "pair_coverage_qc": NOTEBOOK16_COVERAGE_PATH,
    "run_manifest": NOTEBOOK16_MANIFEST_PATH,
}
for key, expected_path in expected_notebook16_outputs.items():
    if Path(notebook16_outputs.get(key, "")) != expected_path:
        raise RuntimeError(f"Notebook 16 output path mismatch: {key}")

if Path(notebook15_manifest.get("notebook14_manifest_path", "")) != NOTEBOOK14_MANIFEST_PATH:
    raise RuntimeError("Notebook 15 and Notebook 26 do not share the same Notebook 14 manifest.")
if Path(
    notebook16_manifest.get("input_paths", {}).get("pipeline_manifest", "")
) != NOTEBOOK14_MANIFEST_PATH:
    raise RuntimeError("Notebook 16 and Notebook 26 do not share the same Notebook 14 manifest.")

print("Manifest and lineage validation: PASS")


Manifest and lineage validation: PASS


In [23]:
# ==== Load Only Notebook 15 Summaries and Notebook 16 Inference Artifacts ====
notebook15_overall = pd.read_csv(NOTEBOOK15_OVERALL_PATH)
notebook15_by_regime = pd.read_csv(NOTEBOOK15_BY_REGIME_PATH)
notebook16_overall = pd.read_csv(NOTEBOOK16_OVERALL_PATH)
notebook16_by_regime = pd.read_csv(NOTEBOOK16_BY_REGIME_PATH)
notebook16_coverage = pd.read_csv(NOTEBOOK16_COVERAGE_PATH)
notebook16_case_deltas = pd.read_parquet(NOTEBOOK16_DELTAS_PATH)

require_columns(
    notebook15_overall,
    [
        "category_id", "reranker_family", "stage_condition",
        "candidate_pool_depth", "metric_name", "metric_cutoff",
        "case_count", "metric_mean",
    ],
    "Notebook 15 overall descriptive summary",
)
require_columns(
    notebook15_by_regime,
    [
        "category_id", "reranker_family", "stage_condition", "regime",
        "candidate_pool_depth", "metric_name", "metric_cutoff",
        "case_count", "metric_mean",
    ],
    "Notebook 15 by-regime descriptive summary",
)
inference_required = [
    "category_id", "comparison_order", "comparison_id", "comparison_label",
    "comparison_method_family", "left_condition", "right_condition",
    "method_match_rule", "method_family_match_pass", "scope", "scope_order",
    "metric_name", "metric_cutoff", "candidate_pool_depth",
    "n_left", "n_right", "n_pairs", "missing_left", "missing_right",
    "mean_left", "mean_right", "mean_delta",
    "bootstrap_ci_95_lower", "bootstrap_ci_95_upper",
    "bootstrap_iterations", "bootstrap_unit",
    "paired_sign_flip_permutation_p_value", "holm_adjusted_p_value",
    "holm_reject_0_05", "positive_delta_count", "positive_delta_rate",
    "tie_count", "tie_rate", "negative_delta_count", "negative_delta_rate",
]
require_columns(notebook16_overall, inference_required, "Notebook 16 overall inference")
require_columns(notebook16_by_regime, inference_required, "Notebook 16 regime inference")
require_columns(
    notebook16_coverage,
    [
        "category_id", "comparison_id", "scope", "n_left", "n_right",
        "n_pairs", "missing_left", "missing_right",
        "query_id_mismatch_count", "user_id_mismatch_count",
        "regime_mismatch_count", "target_item_mismatch_count",
        "coverage_status",
    ],
    "Notebook 16 pair coverage",
)
require_columns(
    notebook16_case_deltas,
    [
        "category_id", "comparison_id", "comparison_method_family",
        "method_family_match_pass", "case_id", "query_id", "user_id", "regime",
        "metric_name", "metric_cutoff", "candidate_pool_depth",
        "left_metric_value", "right_metric_value", "delta",
    ],
    "Notebook 16 paired case deltas",
)
for label, frame in [
    ("Notebook 15 overall", notebook15_overall),
    ("Notebook 15 by-regime", notebook15_by_regime),
    ("Notebook 16 overall", notebook16_overall),
    ("Notebook 16 by-regime", notebook16_by_regime),
    ("Notebook 16 coverage", notebook16_coverage),
    ("Notebook 16 case deltas", notebook16_case_deltas),
]:
    require_unique_columns(frame, label)
    if set(frame["category_id"].astype(str)) != {CATEGORY_ID}:
        raise RuntimeError(f"{label} contains a different category.")

for frame in [notebook15_overall, notebook15_by_regime]:
    frame["reranker_family"] = normalize_lower(frame["reranker_family"])
    frame["stage_condition"] = frame["stage_condition"].astype(str).map(normalize_text)
    frame["metric_name"] = frame["metric_name"].astype(str).map(normalize_text)
    frame["candidate_pool_depth"] = pd.to_numeric(
        frame["candidate_pool_depth"], errors="raise"
    ).astype(int)
    frame["metric_cutoff"] = pd.to_numeric(
        frame["metric_cutoff"], errors="raise"
    ).astype(int)
    frame["case_count"] = pd.to_numeric(
        frame["case_count"], errors="raise"
    ).astype(int)
    frame["metric_mean"] = pd.to_numeric(
        frame["metric_mean"], errors="raise"
    ).astype(float)
notebook15_by_regime["regime"] = normalize_lower(
    notebook15_by_regime["regime"]
)

notebook16_inference = pd.concat(
    [notebook16_overall, notebook16_by_regime],
    ignore_index=True,
    sort=False,
)
for frame in [
    notebook16_inference, notebook16_coverage, notebook16_case_deltas
]:
    if "scope" in frame.columns:
        frame["scope"] = normalize_lower(frame["scope"])
    if "regime" in frame.columns:
        frame["regime"] = normalize_lower(frame["regime"])
    if "comparison_method_family" in frame.columns:
        frame["comparison_method_family"] = normalize_lower(
            frame["comparison_method_family"]
        )

for label, frame in [
    ("Notebook 16 inference", notebook16_inference),
    ("Notebook 16 case deltas", notebook16_case_deltas),
]:
    if not frame["metric_name"].astype(str).eq(PRIMARY_METRIC_NAME).all():
        raise RuntimeError(f"{label} includes a metric other than NDCG.")
    if not pd.to_numeric(frame["metric_cutoff"], errors="raise").eq(
        PRIMARY_METRIC_CUTOFF
    ).all():
        raise RuntimeError(f"{label} includes a cutoff other than 5.")
    if not pd.to_numeric(frame["candidate_pool_depth"], errors="raise").eq(
        REPORT_POOL_DEPTH
    ).all():
        raise RuntimeError(
            f"{label} includes a pool depth other than {REPORT_POOL_DEPTH}."
        )

if "confirmatory_holm_included" in notebook16_by_regime.columns:
    included = parse_bool_series(notebook16_by_regime["confirmatory_holm_included"], "regime confirmatory_holm_included")
    if included.any():
        raise RuntimeError("Regime-specific rows must remain outside the confirmatory Holm family.")
if "holm_adjusted_p_value" in notebook16_by_regime.columns:
    adjusted = pd.to_numeric(notebook16_by_regime["holm_adjusted_p_value"], errors="coerce")
    if adjusted.notna().any():
        raise RuntimeError("Regime-specific rows must not carry Holm-adjusted p-values.")
    notebook16_by_regime["holm_adjusted_p_value"] = np.nan
if "holm_reject_0_05" in notebook16_by_regime.columns:
    reject_status = parse_bool_series(notebook16_by_regime["holm_reject_0_05"], "regime holm_reject_0_05")
    if reject_status.any():
        raise RuntimeError("Regime-specific rows must not carry Holm rejection status.")
    notebook16_by_regime["holm_reject_0_05"] = False
notebook16_by_regime["regime_inference_multiplicity_status"] = "exploratory_unadjusted_no_holm"
if set(notebook16_inference["scope"]) != set(REGIME_SCOPES):
    raise RuntimeError("Notebook 16 inference scopes do not match Notebook 26.")
if notebook16_inference.duplicated(["comparison_id", "scope"]).any():
    raise RuntimeError("Notebook 16 inference rows are duplicated by comparison × scope.")
if notebook16_coverage.duplicated(["comparison_id", "scope"]).any():
    raise RuntimeError("Notebook 16 coverage rows are duplicated by comparison × scope.")
if notebook16_case_deltas.duplicated(["case_id", "comparison_id"]).any():
    raise RuntimeError("Notebook 16 case deltas are duplicated by case × comparison.")
case_identity_counts = notebook16_case_deltas.groupby("case_id")[[
    "query_id", "user_id", "regime"
]].nunique(dropna=False)
if case_identity_counts.gt(1).any().any():
    raise RuntimeError(
        "Notebook 16 case_id does not retain one query_id, user_id, and regime."
    )

print("Input schema and fixed-contract validation: PASS")


Input schema and fixed-contract validation: PASS


## Results

### 1. Condition Means

Overall, cold, weak, moderate, and strong means are direct Notebook 15 summaries. The non-cold mean is the case-count-weighted combination of the three non-cold regime cells. Every resulting condition mean is reconciled with the corresponding Notebook 16 paired-condition mean before export.

In [24]:
# ==== Build One Descriptive Condition Mean Per Reranker × Scope × Condition ====
contract_filter_overall = (
    notebook15_overall["candidate_pool_depth"].eq(REPORT_POOL_DEPTH)
    & notebook15_overall["metric_name"].eq(PRIMARY_METRIC_NAME)
    & notebook15_overall["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF)
    & notebook15_overall["reranker_family"].isin(RERANKER_FAMILIES)
    & notebook15_overall["stage_condition"].isin(
        ["P0", "P1-only", "P2-Q", "P2-P", "b02", "Full"]
    )
)
contract_filter_regime = (
    notebook15_by_regime["candidate_pool_depth"].eq(REPORT_POOL_DEPTH)
    & notebook15_by_regime["metric_name"].eq(PRIMARY_METRIC_NAME)
    & notebook15_by_regime["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF)
    & notebook15_by_regime["reranker_family"].isin(RERANKER_FAMILIES)
    & notebook15_by_regime["stage_condition"].isin(
        ["P0", "P1-only", "P2-Q", "P2-P", "b02", "Full"]
    )
)
direct_overall = notebook15_overall.loc[
    contract_filter_overall,
    ["reranker_family", "stage_condition", "case_count", "metric_mean"],
].copy()
direct_overall["regime"] = "overall"
direct_overall["descriptive_cell_source"] = "Notebook 15 overall summary"
direct_overall["source_regime_count"] = 1

direct_regimes = notebook15_by_regime.loc[
    contract_filter_regime
    & notebook15_by_regime["regime"].isin(CORE_REGIMES),
    [
        "reranker_family", "stage_condition", "regime",
        "case_count", "metric_mean",
    ],
].copy()
direct_regimes["descriptive_cell_source"] = "Notebook 15 by-regime summary"
direct_regimes["source_regime_count"] = 1

non_cold_source = notebook15_by_regime.loc[
    contract_filter_regime
    & notebook15_by_regime["regime"].ne("cold"),
    [
        "reranker_family", "stage_condition", "regime",
        "case_count", "metric_mean",
    ],
].copy()
if non_cold_source.empty:
    raise RuntimeError("Notebook 15 has no non-cold descriptive cells.")
non_cold_source["weighted_metric_sum"] = (
    non_cold_source["case_count"] * non_cold_source["metric_mean"]
)
non_cold = (
    non_cold_source.groupby(
        ["reranker_family", "stage_condition"],
        sort=False,
        observed=True,
    )
    .agg(
        case_count=("case_count", "sum"),
        weighted_metric_sum=("weighted_metric_sum", "sum"),
        source_regime_count=("regime", "nunique"),
    )
    .reset_index()
)
non_cold["metric_mean"] = (
    non_cold["weighted_metric_sum"] / non_cold["case_count"]
)
non_cold = non_cold.drop(columns="weighted_metric_sum")
non_cold["regime"] = "non-cold"
non_cold["descriptive_cell_source"] = (
    "Notebook 15 case-count-weighted non-cold regime summaries"
)

condition_means = pd.concat(
    [direct_overall, direct_regimes, non_cold],
    ignore_index=True,
    sort=False,
)
condition_key = ["reranker_family", "regime", "stage_condition"]
if condition_means.duplicated(condition_key).any():
    raise RuntimeError("Notebook 15 condition means are duplicated.")
EXPECTED_CONDITION_MEAN_CONDITIONS = ["P0", "P1-only", "P2-Q", "P2-P", "b02", "Full"]
expected_condition_rows = (
    len(RERANKER_FAMILIES) * len(REGIME_SCOPES) * len(EXPECTED_CONDITION_MEAN_CONDITIONS)
)
if len(condition_means) != expected_condition_rows:
    raise RuntimeError(
        f"Unexpected Notebook 15 condition-mean row count: {len(condition_means)}"
    )
if condition_means[["case_count", "metric_mean"]].isna().any().any():
    raise RuntimeError("Notebook 15 condition means contain missing values.")

display(
    condition_means.sort_values(
        ["reranker_family", "regime", "stage_condition"],
        kind="mergesort",
    ).head(15)
)


,reranker_family,stage_condition,case_count,metric_mean,regime,descriptive_cell_source,source_regime_count
32,lightgbm,Full,572,0.020043,cold,Notebook 15 by-regime summary,1
12,lightgbm,P0,572,0.001779,cold,Notebook 15 by-regime summary,1
16,lightgbm,P1-only,572,0.001779,cold,Notebook 15 by-regime summary,1
24,lightgbm,P2-P,572,0.020043,cold,Notebook 15 by-regime summary,1
20,lightgbm,P2-Q,572,0.020043,cold,Notebook 15 by-regime summary,1
28,lightgbm,b02,572,0.020043,cold,Notebook 15 by-regime summary,1
33,lightgbm,Full,572,0.051566,moderate,Notebook 15 by-regime summary,1
13,lightgbm,P0,572,0.008256,moderate,Notebook 15 by-regime summary,1
17,lightgbm,P1-only,572,0.011784,moderate,Notebook 15 by-regime summary,1
25,lightgbm,P2-P,572,0.060901,moderate,Notebook 15 by-regime summary,1


### 2. Method-matched regime effect summary

The Notebook 16 inference rows are expanded only for the shared retrieval contrast’s presentation. No new paired confidence interval, sign-flip p-value, or Holm correction is calculated here. Condition means and pair counts must reconcile exactly before export.


In [25]:
# ==== Expand Notebook 16 to Two Presentation Rerankers and Reconcile Sources ====
inference_frames = []
coverage_frames = []
for reranker_family in RERANKER_FAMILIES:
    for comparison in COMPARISON_DEFINITIONS:
        source_id = source_comparison_id(reranker_family, comparison)
        inference_part = notebook16_inference.loc[
            notebook16_inference["comparison_id"].eq(source_id)
        ].copy()
        coverage_part = notebook16_coverage.loc[
            notebook16_coverage["comparison_id"].eq(source_id)
        ].copy()
        if set(inference_part["scope"]) != set(REGIME_SCOPES):
            raise RuntimeError(
                f"Notebook 16 inference scopes are incomplete for {source_id}."
            )
        if set(coverage_part["scope"]) != set(REGIME_SCOPES):
            raise RuntimeError(
                f"Notebook 16 coverage scopes are incomplete for {source_id}."
            )
        if not inference_part["comparison_label"].eq(
            comparison["comparison_label"]
        ).all():
            raise RuntimeError(f"Comparison label mismatch for {source_id}.")
        if not inference_part["left_condition"].eq(
            comparison["left_condition"]
        ).all() or not inference_part["right_condition"].eq(
            comparison["right_condition"]
        ).all():
            raise RuntimeError(f"Comparison condition mismatch for {source_id}.")

        inference_part = inference_part.rename(
            columns={
                "comparison_id": "notebook16_comparison_id",
                "comparison_order": "notebook16_comparison_order",
                "scope": "regime",
                "mean_left": "notebook16_mean_left",
                "mean_right": "notebook16_mean_right",
            }
        )
        inference_part["reranker_family"] = reranker_family
        inference_part["reranker_label"] = RERANKER_LABELS[reranker_family]
        inference_part["comparison_order"] = comparison["comparison_order"]
        inference_part["comparison_id"] = comparison["comparison_id"]
        inference_part["effect_label"] = comparison["effect_label"]
        inference_part["shared_retrieval_effect_repeated_for_reranker_display"] = bool(
            comparison["shared_retrieval_effect"]
        )
        inference_frames.append(inference_part)

        coverage_part = coverage_part.rename(
            columns={
                "comparison_id": "notebook16_comparison_id",
                "scope": "regime",
                "n_left": "coverage_n_left",
                "n_right": "coverage_n_right",
                "n_pairs": "coverage_n_pairs",
                "missing_left": "coverage_missing_left",
                "missing_right": "coverage_missing_right",
                "coverage_status": "notebook16_coverage_status",
            }
        )
        coverage_part["reranker_family"] = reranker_family
        coverage_part["comparison_order"] = comparison["comparison_order"]
        coverage_part["comparison_id"] = comparison["comparison_id"]
        coverage_frames.append(coverage_part)

expanded_inference = pd.concat(
    inference_frames, ignore_index=True, sort=False
)
expanded_coverage = pd.concat(
    coverage_frames, ignore_index=True, sort=False
)
report_key = ["reranker_family", "comparison_id", "regime"]
expected_report_rows = (
    len(RERANKER_FAMILIES)
    * len(COMPARISON_DEFINITIONS)
    * len(REGIME_SCOPES)
)
if len(expanded_inference) != expected_report_rows:
    raise RuntimeError("Expanded Notebook 16 inference row count is unexpected.")
if expanded_inference.duplicated(report_key).any():
    raise RuntimeError("Expanded Notebook 16 inference rows are duplicated.")
if expanded_coverage.duplicated(report_key).any():
    raise RuntimeError("Expanded Notebook 16 coverage rows are duplicated.")

left_means = condition_means.rename(
    columns={
        "stage_condition": "left_condition",
        "case_count": "notebook15_left_case_count",
        "metric_mean": "mean_left_condition",
        "descriptive_cell_source": "left_descriptive_cell_source",
        "source_regime_count": "left_source_regime_count",
    }
)
right_means = condition_means.rename(
    columns={
        "stage_condition": "right_condition",
        "case_count": "notebook15_right_case_count",
        "metric_mean": "mean_right_condition",
        "descriptive_cell_source": "right_descriptive_cell_source",
        "source_regime_count": "right_source_regime_count",
    }
)
regime_effect_summary = expanded_inference.merge(
    left_means,
    on=["reranker_family", "regime", "left_condition"],
    how="left",
    validate="many_to_one",
).merge(
    right_means,
    on=["reranker_family", "regime", "right_condition"],
    how="left",
    validate="many_to_one",
).merge(
    expanded_coverage,
    on=[
        "category_id", "reranker_family", "comparison_order",
        "comparison_id", "notebook16_comparison_id", "regime",
    ],
    how="left",
    validate="one_to_one",
    suffixes=("", "_coverage"),
)
if len(regime_effect_summary) != expected_report_rows:
    raise RuntimeError("Source reconciliation multiplied or dropped report rows.")

regime_effect_summary["mean_left_reconciliation_passed"] = np.isclose(
    regime_effect_summary["mean_left_condition"],
    regime_effect_summary["notebook16_mean_left"],
    rtol=0.0,
    atol=1e-12,
)
regime_effect_summary["mean_right_reconciliation_passed"] = np.isclose(
    regime_effect_summary["mean_right_condition"],
    regime_effect_summary["notebook16_mean_right"],
    rtol=0.0,
    atol=1e-12,
)
regime_effect_summary["mean_delta_reconciliation_passed"] = np.isclose(
    regime_effect_summary["mean_left_condition"]
    - regime_effect_summary["mean_right_condition"],
    regime_effect_summary["mean_delta"],
    rtol=0.0,
    atol=1e-12,
)
regime_effect_summary["left_case_count_reconciliation_passed"] = (
    regime_effect_summary["notebook15_left_case_count"].eq(
        regime_effect_summary["n_left"]
    )
)
regime_effect_summary["right_case_count_reconciliation_passed"] = (
    regime_effect_summary["notebook15_right_case_count"].eq(
        regime_effect_summary["n_right"]
    )
)
regime_effect_summary["coverage_count_reconciliation_passed"] = (
    regime_effect_summary["n_left"].eq(
        regime_effect_summary["coverage_n_left"]
    )
    & regime_effect_summary["n_right"].eq(
        regime_effect_summary["coverage_n_right"]
    )
    & regime_effect_summary["n_pairs"].eq(
        regime_effect_summary["coverage_n_pairs"]
    )
    & regime_effect_summary["missing_left"].eq(
        regime_effect_summary["coverage_missing_left"]
    )
    & regime_effect_summary["missing_right"].eq(
        regime_effect_summary["coverage_missing_right"]
    )
)
regime_effect_summary["case_count"] = regime_effect_summary["n_pairs"].astype(int)
regime_effect_summary["unadjusted_sign_flip_p_value"] = (
    regime_effect_summary["paired_sign_flip_permutation_p_value"].astype(float)
)
regime_effect_summary["regime_order"] = regime_effect_summary["regime"].map(
    REGIME_ORDER
).astype(int)
regime_effect_summary["pair_coverage_status"] = np.where(
    regime_effect_summary["notebook16_coverage_status"].eq("PASS")
    & regime_effect_summary["coverage_count_reconciliation_passed"]
    & regime_effect_summary["left_case_count_reconciliation_passed"]
    & regime_effect_summary["right_case_count_reconciliation_passed"]
    & regime_effect_summary["mean_left_reconciliation_passed"]
    & regime_effect_summary["mean_right_reconciliation_passed"]
    & regime_effect_summary["mean_delta_reconciliation_passed"],
    "PASS",
    "FAIL",
)
regime_effect_summary["comparison_design"] = "case_id_paired"
regime_effect_summary["descriptive_source"] = (
    "Notebook 15 stage-allocation summaries"
)
regime_effect_summary["inference_source"] = (
    "Notebook 16 method-matched paired inference"
)

REGIME_EFFECT_COLUMNS = [
    "category_id", "category_label", "reranker_family", "reranker_label",
    "comparison_order", "comparison_id", "comparison_label", "effect_label",
    "notebook16_comparison_id", "left_condition", "right_condition",
    "shared_retrieval_effect_repeated_for_reranker_display",
    "regime", "regime_order", "metric_name", "metric_cutoff",
    "candidate_pool_depth", "mean_left_condition", "mean_right_condition",
    "mean_delta", "bootstrap_ci_95_lower", "bootstrap_ci_95_upper",
    "unadjusted_sign_flip_p_value", "holm_adjusted_p_value",
    "holm_reject_0_05", "positive_delta_count", "positive_delta_rate",
    "tie_count", "tie_rate", "negative_delta_count", "negative_delta_rate",
    "case_count", "n_left", "n_right", "n_pairs", "missing_left",
    "missing_right", "pair_coverage_status", "comparison_design",
    "method_match_rule", "method_family_match_pass", "bootstrap_iterations",
    "bootstrap_unit", "left_descriptive_cell_source",
    "right_descriptive_cell_source", "descriptive_source", "inference_source",
    "mean_left_reconciliation_passed", "mean_right_reconciliation_passed",
    "mean_delta_reconciliation_passed",
]
regime_effect_summary_report_depth = regime_effect_summary[
    REGIME_EFFECT_COLUMNS
].sort_values(
    ["reranker_family", "comparison_order", "regime_order"],
    kind="mergesort",
).reset_index(drop=True)

coverage_qc = regime_effect_summary.copy()
coverage_qc["coverage_status"] = coverage_qc["pair_coverage_status"]
REGIME_COVERAGE_COLUMNS = [
    "category_id", "reranker_family", "comparison_order", "comparison_id",
    "comparison_label", "notebook16_comparison_id", "regime",
    "metric_name", "metric_cutoff", "candidate_pool_depth",
    "n_left", "n_right", "n_pairs", "missing_left", "missing_right",
    "notebook15_left_case_count", "notebook15_right_case_count",
    "query_id_mismatch_count", "user_id_mismatch_count",
    "regime_mismatch_count", "target_item_mismatch_count",
    "notebook16_coverage_status", "left_case_count_reconciliation_passed",
    "right_case_count_reconciliation_passed",
    "coverage_count_reconciliation_passed",
    "mean_left_reconciliation_passed", "mean_right_reconciliation_passed",
    "mean_delta_reconciliation_passed", "method_family_match_pass",
    "coverage_status",
]
regime_pair_coverage_qc = coverage_qc[REGIME_COVERAGE_COLUMNS].sort_values(
    ["reranker_family", "comparison_order", "regime"],
    kind="mergesort",
).reset_index(drop=True)

if not regime_effect_summary_report_depth["pair_coverage_status"].eq("PASS").all():
    raise RuntimeError("A regime effect row failed source or pair reconciliation.")

display(regime_effect_summary_report_depth.head(15))


,category_id,category_label,reranker_family,reranker_label,comparison_order,comparison_id,comparison_label,effect_label,notebook16_comparison_id,left_condition,right_condition,shared_retrieval_effect_repeated_for_reranker_display,regime,regime_order,metric_name,metric_cutoff,candidate_pool_depth,mean_left_condition,mean_right_condition,mean_delta,bootstrap_ci_95_lower,bootstrap_ci_95_upper,unadjusted_sign_flip_p_value,holm_adjusted_p_value,holm_reject_0_05,positive_delta_count,positive_delta_rate,tie_count,tie_rate,negative_delta_count,negative_delta_rate,case_count,n_left,n_right,n_pairs,missing_left,missing_right,pair_coverage_status,comparison_design,method_match_rule,method_family_match_pass,bootstrap_iterations,bootstrap_unit,left_descriptive_cell_source,right_descriptive_cell_source,descriptive_source,inference_source,mean_left_reconciliation_passed,mean_right_reconciliation_passed,mean_delta_reconciliation_passed
0,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,P1-only,P0,True,overall,0,NDCG,5,1000,0.010081,0.008468,0.001613,-0.000383,0.003674,0.117788,NaN,False,15,0.006556,2264,0.989510,9,0.003934,2288,2288,2288,2288,0,0,PASS,case_id_paired,shared_retrieval_only,True,10000,paired_case_id_user_id_within_regime_strata,Notebook 15 overall summary,Notebook 15 overall summary,Notebook 15 stage-allocation summaries,Notebook 16 method-matched paired inference,True,True,True
1,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,P1-only,P0,True,cold,1,NDCG,5,1000,0.001779,0.001779,0.000000,0.000000,0.000000,1.000000,NaN,False,0,0.000000,572,1.000000,0,0.000000,572,572,572,572,0,0,PASS,case_id_paired,shared_retrieval_only,True,10000,paired_case_id_user_id_within_regime_strata,Notebook 15 by-regime summary,Notebook 15 by-regime summary,Notebook 15 stage-allocation summaries,Notebook 16 method-matched paired inference,True,True,True
2,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,P1-only,P0,True,weak,2,NDCG,5,1000,0.003254,0.003375,-0.000121,-0.005124,0.004007,1.000000,NaN,False,3,0.005245,567,0.991259,2,0.003497,572,572,572,572,0,0,PASS,case_id_paired,shared_retrieval_only,True,10000,paired_case_id_user_id_within_regime_strata,Notebook 15 by-regime summary,Notebook 15 by-regime summary,Notebook 15 stage-allocation summaries,Notebook 16 method-matched paired inference,True,True,True
3,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,P1-only,P0,True,moderate,3,NDCG,5,1000,0.011784,0.008256,0.003529,0.000403,0.007476,0.081692,NaN,False,5,0.008741,565,0.987762,2,0.003497,572,572,572,572,0,0,PASS,case_id_paired,shared_retrieval_only,True,10000,paired_case_id_user_id_within_regime_strata,Notebook 15 by-regime summary,Notebook 15 by-regime summary,Notebook 15 stage-allocation summaries,Notebook 16 method-matched paired inference,True,True,True
4,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,P1-only,P0,True,strong,4,NDCG,5,1000,0.023508,0.020462,0.003046,-0.002296,0.008887,0.285471,NaN,False,7,0.012238,560,0.979021,5,0.008741,572,572,572,572,0,0,PASS,case_id_paired,shared_retrieval_only,True,10000,paired_case_id_user_id_within_regime_strata,Notebook 15 by-regime summary,Notebook 15 by-regime summary,Notebook 15 stage-allocation summaries,Notebook 16 method-matched paired inference,True,True,True
5,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,P1-only,P0,True,non-cold,5,NDCG,5,1000,0.012849,0.010698,0.002151,-0.000471,0.004907,0.116488,NaN,False,15,0.008741,1692,0.986014,9,0.005245,1716,1716,1716,1716,0,0,PASS,case_id_paired,shared_retrieval_only,True,10000,paired_case_id_user_id_within_regime_strata,Notebook 15 case-c

### 3. Strong–Weak heterogeneity

Strong and Weak contain different users. For each fixed reranker and contrast, cases are resampled independently within the Strong and Weak strata. The bootstrap statistic is:

$$D^{*(b)} = \overline{d}^{*(b)}_{Strong} - \overline{d}^{*(b)}_{Weak}.$$

`heterogeneity_supported` is true only when the percentile 95% interval excludes zero. “Strong-regime reversal” is used only as a descriptive label when the observed Strong mean delta is smaller than the Weak mean delta.


In [26]:
# ==== Independent Strong–Weak Bootstrap from Notebook 16 Case Deltas ====
strong_weak_rows = []
for reranker_family in RERANKER_FAMILIES:
    for comparison in COMPARISON_DEFINITIONS:
        source_id = source_comparison_id(reranker_family, comparison)
        source_cases = notebook16_case_deltas.loc[
            notebook16_case_deltas["comparison_id"].eq(source_id)
        ].copy()
        if source_cases.empty:
            raise RuntimeError(f"No Notebook 16 case deltas for {source_id}.")
        if source_cases["case_id"].duplicated().any():
            raise RuntimeError(f"Duplicated case deltas for {source_id}.")
        if not parse_bool_series(
            source_cases["method_family_match_pass"],
            f"method_family_match_pass / {source_id}",
        ).all():
            raise RuntimeError(f"Method-family match failed for {source_id}.")

        strong_cases = source_cases.loc[
            source_cases["regime"].eq("strong")
        ].copy()
        weak_cases = source_cases.loc[
            source_cases["regime"].eq("weak")
        ].copy()
        if strong_cases.empty or weak_cases.empty:
            raise RuntimeError(f"Strong or Weak cases are absent for {source_id}.")
        strong_case_overlap = set(strong_cases["case_id"]).intersection(
            set(weak_cases["case_id"])
        )
        strong_user_overlap = set(strong_cases["user_id"]).intersection(
            set(weak_cases["user_id"])
        )
        if strong_case_overlap or strong_user_overlap:
            raise RuntimeError(
                f"Strong and Weak groups overlap for {source_id}."
            )
        if strong_cases["user_id"].duplicated().any() or weak_cases[
            "user_id"
        ].duplicated().any():
            raise RuntimeError(
                f"The one-case-per-user contract failed for {source_id}."
            )

        strong_values = pd.to_numeric(
            strong_cases["delta"], errors="raise"
        ).to_numpy(dtype=float)
        weak_values = pd.to_numeric(
            weak_cases["delta"], errors="raise"
        ).to_numpy(dtype=float)
        bootstrap_seed = stable_seed(
            CATEGORY_ID, reranker_family, comparison["comparison_id"],
            "strong_weak_independent",
        )
        ci_lower, ci_upper = independent_stratified_bootstrap_mean_difference(
            strong_values, weak_values, bootstrap_seed
        )
        strong_mean = float(strong_values.mean())
        weak_mean = float(weak_values.mean())
        difference = strong_mean - weak_mean
        supported = bool(ci_lower > 0.0 or ci_upper < 0.0)
        if difference > DELTA_TIE_ATOL:
            direction = "strong_larger_than_weak"
        elif difference < -DELTA_TIE_ATOL:
            direction = "strong_smaller_than_weak"
        else:
            direction = "equal_within_tolerance"
        descriptive_label = (
            "supported_strong_below_weak"
            if supported and difference < -DELTA_TIE_ATOL
            else (
                "strong_below_weak_descriptive"
                if difference < -DELTA_TIE_ATOL
                else "no_supported_strong_history_premium"
            )
        )

        report_strong = regime_effect_summary_report_depth.loc[
            regime_effect_summary_report_depth["reranker_family"].eq(
                reranker_family
            )
            & regime_effect_summary_report_depth["comparison_id"].eq(
                comparison["comparison_id"]
            )
            & regime_effect_summary_report_depth["regime"].eq("strong")
        ]
        report_weak = regime_effect_summary_report_depth.loc[
            regime_effect_summary_report_depth["reranker_family"].eq(
                reranker_family
            )
            & regime_effect_summary_report_depth["comparison_id"].eq(
                comparison["comparison_id"]
            )
            & regime_effect_summary_report_depth["regime"].eq("weak")
        ]
        if len(report_strong) != 1 or len(report_weak) != 1:
            raise RuntimeError("Strong–Weak reconciliation rows are not unique.")

        strong_weak_rows.append({
            "category_id": CATEGORY_ID,
            "category_label": CATEGORY_LABEL,
            "reranker_family": reranker_family,
            "reranker_label": RERANKER_LABELS[reranker_family],
            "comparison_order": comparison["comparison_order"],
            "comparison_id": comparison["comparison_id"],
            "comparison_label": comparison["comparison_label"],
            "effect_label": comparison["effect_label"],
            "notebook16_comparison_id": source_id,
            "metric_name": PRIMARY_METRIC_NAME,
            "metric_cutoff": PRIMARY_METRIC_CUTOFF,
            "candidate_pool_depth": REPORT_POOL_DEPTH,
            "strong_mean_delta": strong_mean,
            "weak_mean_delta": weak_mean,
            "strong_minus_weak_delta": difference,
            "bootstrap_ci_95_lower": ci_lower,
            "bootstrap_ci_95_upper": ci_upper,
            "strong_case_count": int(len(strong_cases)),
            "weak_case_count": int(len(weak_cases)),
            "strong_user_count": int(strong_cases["user_id"].nunique()),
            "weak_user_count": int(weak_cases["user_id"].nunique()),
            "strong_weak_case_overlap_count": int(len(strong_case_overlap)),
            "strong_weak_user_overlap_count": int(len(strong_user_overlap)),
            "bootstrap_iterations": STRONG_WEAK_BOOTSTRAP_ITERATIONS,
            "bootstrap_seed": int(bootstrap_seed),
            "bootstrap_unit": "case_id",
            "bootstrap_design": "independent_resampling_within_strong_and_weak_strata",
            "comparison_design": "independent_regime_groups_not_paired",
            "strong_vs_weak_direction": direction,
            "heterogeneity_supported": supported,
            "evidence_status": (
                "interval_excludes_zero" if supported else "descriptive_only"
            ),
            "descriptive_label": descriptive_label,
            "strong_mean_reconciliation_passed": bool(np.isclose(
                strong_mean,
                float(report_strong.iloc[0]["mean_delta"]),
                rtol=0.0,
                atol=1e-12,
            )),
            "weak_mean_reconciliation_passed": bool(np.isclose(
                weak_mean,
                float(report_weak.iloc[0]["mean_delta"]),
                rtol=0.0,
                atol=1e-12,
            )),
        })

strong_weak_difference_in_delta = pd.DataFrame(strong_weak_rows).sort_values(
    ["reranker_family", "comparison_order"], kind="mergesort"
).reset_index(drop=True)
if not strong_weak_difference_in_delta[
    ["strong_mean_reconciliation_passed", "weak_mean_reconciliation_passed"]
].all().all():
    raise RuntimeError("Strong–Weak means do not reconcile with Notebook 16.")

display(strong_weak_difference_in_delta)


,category_id,category_label,reranker_family,reranker_label,comparison_order,comparison_id,comparison_label,effect_label,notebook16_comparison_id,metric_name,metric_cutoff,candidate_pool_depth,strong_mean_delta,weak_mean_delta,strong_minus_weak_delta,bootstrap_ci_95_lower,bootstrap_ci_95_upper,strong_case_count,weak_case_count,strong_user_count,weak_user_count,strong_weak_case_overlap_count,strong_weak_user_overlap_count,bootstrap_iterations,bootstrap_seed,bootstrap_unit,bootstrap_design,comparison_design,strong_vs_weak_direction,heterogeneity_supported,evidence_status,descriptive_label,strong_mean_reconciliation_passed,weak_mean_reconciliation_passed
0,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,NDCG,5,1000,0.003046,-0.000121,0.003167,-0.003833,0.010691,572,572,572,572,0,0,10000,366997957,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_larger_than_weak,False,descriptive_only,no_supported_strong_history_premium,True,True
1,face,Facial Skincare,lightgbm,LightGBM,2,P2_Q_minus_P0,P2-Q − P0,No-prior reranking effect,lightgbm_P2_Q_minus_P0,NDCG,5,1000,0.042941,0.031108,0.011833,-0.010902,0.034651,572,572,572,572,0,0,10000,1198946440,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_larger_than_weak,False,descriptive_only,no_supported_strong_history_premium,True,True
2,face,Facial Skincare,lightgbm,LightGBM,3,P2_P_minus_P2_Q,P2-P − P2-Q,Stage 2 prior-feature effect,lightgbm_P2_P_minus_P2_Q,NDCG,5,1000,0.009965,0.023774,-0.013809,-0.030564,0.002825,572,572,572,572,0,0,10000,4249648518,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_smaller_than_weak,False,descriptive_only,strong_below_weak_descriptive,True,True
3,face,Facial Skincare,lightgbm,LightGBM,4,Full_minus_P2_P,Full − P2-P,Personalized candidate-source effect,lightgbm_Full_minus_P2_P,NDCG,5,1000,-0.000882,-0.006057,0.005174,-0.012844,0.022551,572,572,572,572,0,0,10000,940039147,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_larger_than_weak,False,descriptive_only,no_supported_strong_history_premium,True,True
4,face,Facial Skincare,lightgbm,LightGBM,5,Full_minus_P2_Q,Full − P2-Q,Combined personalization effect,lightgbm_Full_minus_P2_Q,NDCG,5,1000,0.009083,0.017718,-0.008635,-0.027606,0.011100,572,572,572,572,0,0,10000,1773654974,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_smaller_than_weak,False,descriptive_only,strong_below_weak_descriptive,True,True
5,face,Facial Skincare,transformer,Transformer,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,P1_only_minus_P0,NDCG,5,1000,0.003046,-0.000121,0.003167,-0.003749,0.010611,572,572,572,572,0,0,10000,2057458138,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_larger_than_weak,False,descriptive_only,no_supported_strong_history_premium,True,True
6,face,Facial Skincare,transformer,Transformer,2,P2_Q_minus_P0,P2-Q − P0,No-prior reranking effect,transformer_P2_Q_minus_P0,NDCG,5,1000,0.041958,0.013730,0.028228,0.008539,0.048589,572,572,572,572,0,0,10000,1825276513,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_larger_than_weak,True,interval_excludes_zero,no_supported_strong_history_premium,True,True
7,face,Facial Skincare,transformer,Transformer,3,P2_P_minus_P2_Q,P2-P − P2-Q,Stage 2 prior-feature effect,transformer_P2_P_minus_P2_Q,NDCG,5,1000,-0.011447,0.022029,-0.033476,-0.053459,-0.013280,572,572,572,572,0,0,10000,2145847556,case_id,independent_resampling_within_strong_and_weak_...,independent_regime_groups_not_paired,strong_smaller_than_weak,True,interval_excludes_zero,supported_strong_below_weak,True,True
8,face,Facial Skincare,transformer,Transformer,4,Full_minus_P2_P,Full

### 4. Evidence Scorecard

The scorecard records the largest observed mean and absolute mean delta across cold, weak, moderate, and strong; exploratory regime intervals that exclude zero; and the independently bootstrapped Strong-minus-Weak relation. These fields are descriptive and unadjusted. They do not assign pipeline winners, causal mechanisms, or confirmatory status.

In [27]:
# ==== Build a Compact Evidence Scorecard Without Method Selection ====
scorecard_rows = []
for reranker_family in RERANKER_FAMILIES:
    for comparison in COMPARISON_DEFINITIONS:
        cells = regime_effect_summary_report_depth.loc[
            regime_effect_summary_report_depth["reranker_family"].eq(
                reranker_family
            )
            & regime_effect_summary_report_depth["comparison_id"].eq(
                comparison["comparison_id"]
            )
            & regime_effect_summary_report_depth["regime"].isin(
                CORE_REGIMES
            )
        ].copy()
        if set(cells["regime"]) != set(CORE_REGIMES):
            raise RuntimeError("Scorecard regime cells are incomplete.")
        cells["core_regime_order"] = cells["regime"].map(
            {value: index for index, value in enumerate(CORE_REGIMES)}
        )
        largest_mean = cells.sort_values(
            ["mean_delta", "core_regime_order"],
            ascending=[False, True],
            kind="mergesort",
        ).iloc[0]
        largest_absolute = cells.assign(
            absolute_mean_delta=cells["mean_delta"].abs()
        ).sort_values(
            ["absolute_mean_delta", "core_regime_order"],
            ascending=[False, True],
            kind="mergesort",
        ).iloc[0]

        supported_positive = cells.loc[
            cells["bootstrap_ci_95_lower"].gt(0.0)
            & cells["mean_delta"].gt(DELTA_TIE_ATOL),
            "regime",
        ].tolist()
        supported_negative = cells.loc[
            cells["bootstrap_ci_95_upper"].lt(0.0)
            & cells["mean_delta"].lt(-DELTA_TIE_ATOL),
            "regime",
        ].tolist()
        descriptive_positive = cells.loc[
            ~cells["bootstrap_ci_95_lower"].gt(0.0)
            & cells["mean_delta"].gt(DELTA_TIE_ATOL),
            "regime",
        ].tolist()
        descriptive_negative = cells.loc[
            ~cells["bootstrap_ci_95_upper"].lt(0.0)
            & cells["mean_delta"].lt(-DELTA_TIE_ATOL),
            "regime",
        ].tolist()
        heterogeneity = strong_weak_difference_in_delta.loc[
            strong_weak_difference_in_delta["reranker_family"].eq(
                reranker_family
            )
            & strong_weak_difference_in_delta["comparison_id"].eq(
                comparison["comparison_id"]
            )
        ]
        if len(heterogeneity) != 1:
            raise RuntimeError("Scorecard Strong–Weak row is not unique.")
        heterogeneity = heterogeneity.iloc[0]
        if supported_positive or supported_negative:
            evidence_summary = "exploratory_regime_interval_excludes_zero"
        else:
            evidence_summary = "descriptive_regime_pattern_only"

        scorecard_rows.append({
            "category_id": CATEGORY_ID,
            "category_label": CATEGORY_LABEL,
            "reranker_family": reranker_family,
            "reranker_label": RERANKER_LABELS[reranker_family],
            "comparison_order": comparison["comparison_order"],
            "comparison_id": comparison["comparison_id"],
            "comparison_label": comparison["comparison_label"],
            "effect_label": comparison["effect_label"],
            "metric_name": PRIMARY_METRIC_NAME,
            "metric_cutoff": PRIMARY_METRIC_CUTOFF,
            "candidate_pool_depth": REPORT_POOL_DEPTH,
            "regime_multiplicity_status": "exploratory_unadjusted",
            "localization_regimes": "cold|weak|strong",
            "largest_observed_mean_delta_regime": largest_mean["regime"],
            "largest_observed_mean_delta": float(largest_mean["mean_delta"]),
            "largest_observed_absolute_mean_delta_regime": largest_absolute["regime"],
            "largest_observed_absolute_mean_delta": float(
                abs(largest_absolute["mean_delta"])
            ),
            "supported_positive_regimes_unadjusted": "|".join(supported_positive),
            "supported_negative_regimes_unadjusted": "|".join(supported_negative),
            "descriptive_positive_only_regimes": "|".join(
                descriptive_positive
            ),
            "descriptive_negative_only_regimes": "|".join(
                descriptive_negative
            ),
            "strong_vs_weak_direction": heterogeneity[
                "strong_vs_weak_direction"
            ],
            "strong_minus_weak_delta": float(
                heterogeneity["strong_minus_weak_delta"]
            ),
            "strong_weak_ci_95_lower": float(
                heterogeneity["bootstrap_ci_95_lower"]
            ),
            "strong_weak_ci_95_upper": float(
                heterogeneity["bootstrap_ci_95_upper"]
            ),
            "heterogeneity_supported": bool(
                heterogeneity["heterogeneity_supported"]
            ),
            "descriptive_label": heterogeneity["descriptive_label"],
            "evidence_summary": evidence_summary,
        })

regime_evidence_scorecard = pd.DataFrame(scorecard_rows).sort_values(
    ["reranker_family", "comparison_order"], kind="mergesort"
).reset_index(drop=True)
display(regime_evidence_scorecard)


,category_id,category_label,reranker_family,reranker_label,comparison_order,comparison_id,comparison_label,effect_label,metric_name,metric_cutoff,candidate_pool_depth,regime_multiplicity_status,localization_regimes,largest_observed_mean_delta_regime,largest_observed_mean_delta,largest_observed_absolute_mean_delta_regime,largest_observed_absolute_mean_delta,supported_positive_regimes_unadjusted,supported_negative_regimes_unadjusted,descriptive_positive_only_regimes,descriptive_negative_only_regimes,strong_vs_weak_direction,strong_minus_weak_delta,strong_weak_ci_95_lower,strong_weak_ci_95_upper,heterogeneity_supported,descriptive_label,evidence_summary
0,face,Facial Skincare,lightgbm,LightGBM,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,moderate,0.003529,moderate,0.003529,moderate,,strong,weak,strong_larger_than_weak,0.003167,-0.003833,0.010691,False,no_supported_strong_history_premium,exploratory_regime_interval_excludes_zero
1,face,Facial Skincare,lightgbm,LightGBM,2,P2_Q_minus_P0,P2-Q − P0,No-prior reranking effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,strong,0.042941,strong,0.042941,cold|weak|moderate|strong,,,,strong_larger_than_weak,0.011833,-0.010902,0.034651,False,no_supported_strong_history_premium,exploratory_regime_interval_excludes_zero
2,face,Facial Skincare,lightgbm,LightGBM,3,P2_P_minus_P2_Q,P2-P − P2-Q,Stage 2 prior-feature effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,weak,0.023774,weak,0.023774,weak|moderate|strong,,,,strong_smaller_than_weak,-0.013809,-0.030564,0.002825,False,strong_below_weak_descriptive,exploratory_regime_interval_excludes_zero
3,face,Facial Skincare,lightgbm,LightGBM,4,Full_minus_P2_P,Full − P2-P,Personalized candidate-source effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,cold,0.000000,moderate,0.009335,,,,weak|moderate|strong,strong_larger_than_weak,0.005174,-0.012844,0.022551,False,no_supported_strong_history_premium,descriptive_regime_pattern_only
4,face,Facial Skincare,lightgbm,LightGBM,5,Full_minus_P2_Q,Full − P2-Q,Combined personalization effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,weak,0.017718,weak,0.017718,weak,,moderate|strong,,strong_smaller_than_weak,-0.008635,-0.027606,0.011100,False,strong_below_weak_descriptive,exploratory_regime_interval_excludes_zero
5,face,Facial Skincare,transformer,Transformer,1,P1_only_minus_P0,P1-only − P0,Stage 1 personalization effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,moderate,0.003529,moderate,0.003529,moderate,,strong,weak,strong_larger_than_weak,0.003167,-0.003749,0.010611,False,no_supported_strong_history_premium,exploratory_regime_interval_excludes_zero
6,face,Facial Skincare,transformer,Transformer,2,P2_Q_minus_P0,P2-Q − P0,No-prior reranking effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,strong,0.041958,strong,0.041958,cold|weak|moderate|strong,,,,strong_larger_than_weak,0.028228,0.008539,0.048589,True,no_supported_strong_history_premium,exploratory_regime_interval_excludes_zero
7,face,Facial Skincare,transformer,Transformer,3,P2_P_minus_P2_Q,P2-P − P2-Q,Stage 2 prior-feature effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,weak,0.022029,weak,0.022029,weak,,moderate,strong,strong_smaller_than_weak,-0.033476,-0.053459,-0.013280,True,supported_strong_below_weak,exploratory_regime_interval_excludes_zero
8,face,Facial Skincare,transformer,Transformer,4,Full_minus_P2_P,Full − P2-P,Personalized candidate-source effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,weak,0.006449,weak,0.006449,,,weak|strong,moderate,strong_smaller_than_weak,-0.004617,-0.021684,0.013021,False,strong_below_weak_descriptive,descriptive_regime_pattern_only
9,face,Facial Skincare,transformer,Transformer,5,Full_minus_P2_Q,Full − P2-Q,Combined personalization effect,NDCG,5,1000,exploratory_unadjusted,cold|weak|strong,weak,0.028479,weak,0.028479,weak,,moderate,strong,strong_smaller_than_weak,-0.038093,-0.06061

## Checks & Exports

The final gate enforces fixed NDCG@5/depth-1000 rows, exact Notebook 16 reconciliation, one row per category × reranker × regime × comparison, complete pair coverage, and independent Strong–Weak group treatment. Output schemas are recorded in the manifest for cross-category comparison.


In [28]:
# ==== Final validation, exports, and Manifest ====
output_frames = {
    "regime_effect_summary_report_depth": regime_effect_summary_report_depth,
    "strong_weak_difference_in_delta": strong_weak_difference_in_delta,
    "regime_evidence_scorecard": regime_evidence_scorecard,
    "regime_pair_coverage_qc": regime_pair_coverage_qc,
}
for label, frame in output_frames.items():
    require_unique_columns(frame, label)

expected_report_rows = (
    len(RERANKER_FAMILIES)
    * len(COMPARISON_DEFINITIONS)
    * len(REGIME_SCOPES)
)
expected_comparison_rows = len(RERANKER_FAMILIES) * len(
    COMPARISON_DEFINITIONS
)
if len(regime_effect_summary_report_depth) != expected_report_rows:
    raise RuntimeError("Regime effect report has an unexpected row count.")
if len(regime_pair_coverage_qc) != expected_report_rows:
    raise RuntimeError("Regime pair coverage QC has an unexpected row count.")
if len(strong_weak_difference_in_delta) != expected_comparison_rows:
    raise RuntimeError("Strong–Weak output has an unexpected row count.")
if len(regime_evidence_scorecard) != expected_comparison_rows:
    raise RuntimeError("Evidence scorecard has an unexpected row count.")

report_uniqueness_key = [
    "category_id", "reranker_family", "regime", "comparison_id"
]
if regime_effect_summary_report_depth.duplicated(
    report_uniqueness_key
).any():
    raise RuntimeError(
        "Duplicate category × reranker × regime × comparison rows found."
    )
if regime_pair_coverage_qc.duplicated(report_uniqueness_key).any():
    raise RuntimeError("Duplicate regime coverage rows found.")
comparison_uniqueness_key = [
    "category_id", "reranker_family", "comparison_id"
]
if strong_weak_difference_in_delta.duplicated(
    comparison_uniqueness_key
).any():
    raise RuntimeError("Duplicate Strong–Weak comparison rows found.")
if regime_evidence_scorecard.duplicated(
    comparison_uniqueness_key
).any():
    raise RuntimeError("Duplicate scorecard comparison rows found.")

for label, frame in output_frames.items():
    if not frame["metric_name"].eq(PRIMARY_METRIC_NAME).all():
        raise RuntimeError(f"{label} contains a non-NDCG metric.")
    if not frame["metric_cutoff"].eq(PRIMARY_METRIC_CUTOFF).all():
        raise RuntimeError(f"{label} contains a cutoff other than 5.")
    if not frame["candidate_pool_depth"].eq(REPORT_POOL_DEPTH).all():
        raise RuntimeError(
            f"{label} contains a depth other than {REPORT_POOL_DEPTH}."
        )

reconciliation_columns = [
    "mean_left_reconciliation_passed",
    "mean_right_reconciliation_passed",
    "mean_delta_reconciliation_passed",
]
if not regime_effect_summary_report_depth[
    reconciliation_columns
].all().all():
    raise RuntimeError("A Notebook 15/16 effect estimate failed reconciliation.")
if not regime_effect_summary_report_depth["method_family_match_pass"].all():
    raise RuntimeError("A Stage 2 effect is not method-family matched.")
if not regime_pair_coverage_qc["coverage_status"].eq("PASS").all():
    raise RuntimeError("A pair-coverage row failed.")
if (
    strong_weak_difference_in_delta["strong_weak_case_overlap_count"].ne(0).any()
    or strong_weak_difference_in_delta["strong_weak_user_overlap_count"].ne(0).any()
):
    raise RuntimeError("Strong and Weak groups overlap.")
if not strong_weak_difference_in_delta["comparison_design"].eq(
    "independent_regime_groups_not_paired"
).all():
    raise RuntimeError("Strong–Weak comparison was not kept independent.")
if not strong_weak_difference_in_delta["bootstrap_design"].eq(
    "independent_resampling_within_strong_and_weak_strata"
).all():
    raise RuntimeError("Strong–Weak bootstrap design mismatch.")

regime_effect_summary_report_depth.to_csv(
    OUTPUT_FILES["regime_effect_summary_report_depth"],
    index=False,
    encoding="utf-8-sig",
)
strong_weak_difference_in_delta.to_csv(
    OUTPUT_FILES["strong_weak_difference_in_delta"],
    index=False,
    encoding="utf-8-sig",
)
regime_evidence_scorecard.to_csv(
    OUTPUT_FILES["regime_evidence_scorecard"],
    index=False,
    encoding="utf-8-sig",
)
regime_pair_coverage_qc.to_csv(
    OUTPUT_FILES["regime_pair_coverage_qc"],
    index=False,
    encoding="utf-8-sig",
)

validation_results = {
    "notebook_json_validated_before_delivery": True,
    "python_cells_syntax_validated_before_delivery": True,
    "report_depth_1000_fixed_before_input_load": True,
    "ndcg_at_5_fixed_before_input_load": True,
    "notebook15_is_descriptive_authority": True,
    "notebook16_is_delta_and_inference_authority": True,
    "notebook14_manifest_only": True,
    "no_case_metric_reconstruction": True,
    "no_notebook14_descriptive_reaggregation": True,
    "all_effect_estimates_reconcile_with_notebook16": True,
    "one_row_per_category_reranker_regime_comparison": True,
    "all_stage2_effects_method_family_matched": True,
    "case_query_user_regime_grain_preserved": True,
    "shared_retrieval_inference_repeated_for_presentation_only": True,
    "pair_coverage_complete": True,
    "strong_weak_groups_disjoint": True,
    "strong_weak_independent_bootstrap": True,
    "upstream_fold_and_candidate_source_lineage_inherited": True,
    "no_history_or_target_feature_inputs_loaded": True,
    "output_columns_unique": True,
    "required_outputs_written": True,
    "interpretation_limited_to_effect_localization": True,
}
run_manifest = {
    "run_status": "SUCCESS",
    "ready_for_downstream": True,
    "notebook_number": 26,
    "notebook_name": NOTEBOOK_NAME,
    "latest_revision": (
        f"thesis-ready regime-effect summary ({REVISION_DATE})"
    ),
    "category_id": CATEGORY_ID,
    "category_key": CATEGORY_KEY,
    "category_label": CATEGORY_LABEL,
    "analysis_scope": "regime_effect_summary_only",
    "regime_multiplicity_status": "exploratory_unadjusted_outside_locked_holm_family",
    "cold_zero_interpretation": "fallback_integrity_property_not_substantive_finding",
    "cross_category_pooled_significance_test_created": False,
    "input_roles": {
        "notebook15": "authoritative descriptive stage-allocation summaries",
        "notebook16": "authoritative paired deltas and statistical inference; regime slices remain exploratory and unadjusted",
        "notebook14": "manifest provenance and canonical-quality gates only",
    },
    "input_paths": {
        "notebook15_overall": str(NOTEBOOK15_OVERALL_PATH),
        "notebook15_by_regime": str(NOTEBOOK15_BY_REGIME_PATH),
        "notebook15_manifest": str(NOTEBOOK15_MANIFEST_PATH),
        "notebook16_case_deltas": str(NOTEBOOK16_DELTAS_PATH),
        "notebook16_overall_inference": str(NOTEBOOK16_OVERALL_PATH),
        "notebook16_by_regime_inference": str(NOTEBOOK16_BY_REGIME_PATH),
        "notebook16_pair_coverage": str(NOTEBOOK16_COVERAGE_PATH),
        "notebook16_manifest": str(NOTEBOOK16_MANIFEST_PATH),
        "notebook14_manifest": str(NOTEBOOK14_MANIFEST_PATH),
    },
    "primary_evaluation_contract": {
        "candidate_pool_depth": REPORT_POOL_DEPTH,
        "metric_name": PRIMARY_METRIC_NAME,
        "metric_cutoff": PRIMARY_METRIC_CUTOFF,
        "reranker_families": RERANKER_FAMILIES,
        "regime_scopes": REGIME_SCOPES,
    },
    "comparison_definitions": COMPARISON_DEFINITIONS,
    "strong_weak_contract": {
        "estimand": "strong mean case delta minus weak mean case delta",
        "comparison_design": "independent_regime_groups_not_paired",
        "bootstrap_design": "independent_resampling_within_strong_and_weak_strata",
        "bootstrap_unit": "case_id",
        "iterations": STRONG_WEAK_BOOTSTRAP_ITERATIONS,
        "base_seed": STRONG_WEAK_BOOTSTRAP_BASE_SEED,
        "confidence_interval": "percentile_2.5_97.5",
        "support_rule": "95% interval excludes zero",
    },
    "shared_retrieval_presentation_policy": {
        "source_inference_count": 1,
        "presentation_rerankers": RERANKER_FAMILIES,
        "new_test_created": False,
    },
    "row_counts": {
        key: int(len(frame)) for key, frame in output_frames.items()
    },
    "output_paths": {
        key: str(path) for key, path in OUTPUT_FILES.items()
    },
    "output_schemas": {
        key: list(frame.columns) for key, frame in output_frames.items()
    },
    "validation_results": validation_results,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(OUTPUT_FILES["run_manifest"], run_manifest)
reloaded_manifest = load_json(OUTPUT_FILES["run_manifest"])
if reloaded_manifest.get("run_status") != "SUCCESS" or reloaded_manifest.get("ready_for_downstream") is not True:
    raise RuntimeError("Notebook 26 manifest is not ready for downstream use.")

missing_outputs = [
    str(path) for path in OUTPUT_FILES.values() if not path.exists()
]
if missing_outputs:
    raise RuntimeError(f"Required Notebook 26 outputs were not written: {missing_outputs}")

print("Notebook 26 outputs written to:", OUT_DIR)
print("Validation: PASS")


Notebook 26 outputs written to: /content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/regime_analysis_report
Validation: PASS


## Takeaways

Use `regime_effect_summary_report_depth.csv` for stage-effect reporting, `strong_weak_difference_in_delta.csv` for the independent Strong–Weak contrast, and `regime_evidence_scorecard.csv` for a compact evidence map. Any explanation of the observed regime differences belongs in later synthesis notebooks, not in this notebook.


## Validation Tests

In [29]:
# ==== SELF-CHECK SC-5 — Regime Diagnostics Synthetic Failure Tests ====
# SELF-CHECK SC-5: regime diagnostics synthetic failure tests.
def _sc5_regime_diagnostics_synthetic_tests():
    import re
    import numpy as np
    import pandas as pd

    allowed_qchs_labels = {
        "Stage1-QCHS-active",
        "Stage1-QCHS-fallback",
        "Stage2-QCHS-filtered-prior-active",
        "Stage2-QCHS-filtered-prior-inactive",
    }

    def assert_no_bare_qchs_active(labels):
        bad = []
        for label in labels:
            text = str(label)
            bare_label = "QCHS" + "-active"
            bare_pattern = r"(^|[^A-Za-z0-9-])" + "QCHS" + r"-active($|[^A-Za-z0-9-])"
            if text == bare_label or re.search(bare_pattern, text):
                bad.append(text)
        if bad:
            raise AssertionError(f"Bare QCHS active labels are forbidden: {bad}")
        unknown = [str(label) for label in labels if "QCHS" in str(label) and str(label) not in allowed_qchs_labels]
        if unknown:
            raise AssertionError(f"Unexpected QCHS population labels: {unknown}")

    def assert_regime_rows_unadjusted(frame):
        work = frame.copy()
        non_overall = work["scope"].astype(str).str.lower().ne("overall")
        if "confirmatory_holm_included" in work.columns:
            included = work.loc[non_overall, "confirmatory_holm_included"].fillna(False).astype(bool)
            if included.any():
                raise AssertionError("Regime rows must not enter the confirmatory Holm family.")
        if "holm_adjusted_p_value" in work.columns:
            adjusted = pd.to_numeric(work.loc[non_overall, "holm_adjusted_p_value"], errors="coerce")
            if adjusted.notna().any():
                raise AssertionError("Regime rows must not carry Holm-adjusted p-values.")
        if "holm_reject_0_05" in work.columns:
            rejected = work.loc[non_overall, "holm_reject_0_05"].fillna(False).astype(bool)
            if rejected.any():
                raise AssertionError("Regime rows must not carry Holm rejection status.")

    def classify_strong_history(mean_delta, ci_low, ci_high):
        if pd.notna(ci_low) and pd.notna(ci_high) and ci_high < 0:
            return "supported strong below weak"
        if pd.notna(ci_low) and pd.notna(ci_high) and ci_low > 0:
            return "supported reversal"
        if pd.notna(mean_delta) and mean_delta < 0:
            return "descriptive strong below weak"
        return "no supported strong-history premium"

    def assert_strong_history_claim_supported(claim, mean_delta, ci_low, ci_high):
        expected = classify_strong_history(mean_delta, ci_low, ci_high)
        if claim != expected:
            raise AssertionError(
                f"Strong-history claim exceeds CI support: claim={claim}, expected={expected}"
            )

    negative_tests = {}
    for name, fn in {
        "regime_holm_status_fails": lambda: assert_regime_rows_unadjusted(pd.DataFrame([{
            "scope": "strong", "holm_adjusted_p_value": 0.01, "holm_reject_0_05": True,
        }])),
        "bare_qchs_active_fails": lambda: assert_no_bare_qchs_active(["QCHS" + "-active"]),
        "unsupported_strong_history_claim_fails": lambda: assert_strong_history_claim_supported(
            "supported reversal", mean_delta=0.01, ci_low=-0.05, ci_high=0.07
        ),
    }.items():
        try:
            fn()
        except AssertionError:
            negative_tests[name] = True
        else:
            raise AssertionError(f"Synthetic negative test did not fail: {name}")

    assert_regime_rows_unadjusted(pd.DataFrame([{
        "scope": "strong", "holm_adjusted_p_value": np.nan,
        "holm_reject_0_05": False, "confirmatory_holm_included": False,
    }]))
    assert_no_bare_qchs_active(["Stage1-QCHS-active", "Stage2-QCHS-filtered-prior-active"])
    assert_strong_history_claim_supported(
        "descriptive strong below weak", mean_delta=-0.01, ci_low=-0.05, ci_high=0.02
    )

    return {
        **negative_tests,
        "valid_unadjusted_regime_row_passes": True,
        "valid_distinct_qchs_labels_pass": True,
        "valid_descriptive_strong_history_claim_passes": True,
    }

SC5_REGIME_SYNTHETIC_TEST_RESULTS = _sc5_regime_diagnostics_synthetic_tests()
print("SC-5 regime synthetic tests passed:", SC5_REGIME_SYNTHETIC_TEST_RESULTS)

SC-5 regime synthetic tests passed: {'regime_holm_status_fails': True, 'bare_qchs_active_fails': True, 'unsupported_strong_history_claim_fails': True, 'valid_unadjusted_regime_row_passes': True, 'valid_distinct_qchs_labels_pass': True, 'valid_descriptive_strong_history_claim_passes': True}


In [30]:
# ==== Final Verification Report Export ====
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math

try:
    import numpy as _report_np
except Exception:
    _report_np = None
try:
    import pandas as _report_pd
except Exception:
    _report_pd = globals().get("pd")


def _report_is_dataframe(value):
    return _report_pd is not None and isinstance(value, _report_pd.DataFrame)


def _report_is_missing(value):
    if value is None:
        return True
    if isinstance(value, float) and math.isnan(value):
        return True
    if _report_pd is not None:
        try:
            missing = _report_pd.isna(value)
            if isinstance(missing, (bool, type(None))):
                return bool(missing)
        except Exception:
            pass
    return False


def _report_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if _report_np is not None and isinstance(value, _report_np.generic):
        return _report_jsonable(value.item())
    if _report_is_missing(value):
        return None
    if isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float):
        return None if math.isnan(value) else value
    if isinstance(value, dict):
        return {str(k): _report_jsonable(v) for k, v in value.items()}
    if _report_pd is not None:
        if isinstance(value, _report_pd.Series):
            return [_report_jsonable(v) for v in value.tolist()]
        if _report_is_dataframe(value):
            return _report_frame_records(value)
    if isinstance(value, (list, tuple, set)):
        return [_report_jsonable(v) for v in value]
    if hasattr(value, "tolist"):
        try:
            return _report_jsonable(value.tolist())
        except Exception:
            pass
    if hasattr(value, "item"):
        try:
            return _report_jsonable(value.item())
        except Exception:
            pass
    return str(value)


def _report_frame_records(frame, limit=50, columns=None, drop_case_columns=True):
    if not _report_is_dataframe(frame):
        return []
    work = frame.copy()
    if columns is not None:
        keep = [column for column in columns if column in work.columns]
        work = work[keep]
    if drop_case_columns:
        disallowed = {
            "case_id", "query_id", "user_id", "item_id", "parent_asin",
            "target_parent_asin", "target_item_id", "review_id",
        }
        drop = [column for column in work.columns if str(column).lower() in disallowed]
        if drop:
            work = work.drop(columns=drop)
    records = [_report_jsonable(row) for row in work.head(limit).to_dict(orient="records")]
    if len(work) > limit:
        records.append({"note": "truncated", "row_count": int(len(work)), "rows_emitted": int(limit)})
    return records


def _report_output_dir():
    if "OUT_DIR" in globals():
        return Path(globals()["OUT_DIR"])
    if "OUTPUT_DIR" in globals():
        return Path(globals()["OUTPUT_DIR"])
    raise RuntimeError("No existing output-directory variable was found; expected OUT_DIR or OUTPUT_DIR.")


_report_dir = _report_output_dir() / "report"
_report_dir.mkdir(parents=True, exist_ok=True)


def _report_is_inside(path, parent):
    try:
        Path(path).resolve().relative_to(Path(parent).resolve())
        return True
    except Exception:
        return False


def _report_file_sha256(path, allow_heavy=False):
    try:
        path = Path(path)
    except Exception:
        return None
    if not path.exists() or not path.is_file():
        return None
    if not allow_heavy and path.suffix.lower() in {".parquet", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}:
        return None
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _report_notebook_path():
    candidates = []
    for key in ["__vsc_ipynb_file__", "__file__"]:
        value = globals().get(key)
        if value:
            candidates.append(Path(value))
    notebook_name = globals().get("NOTEBOOK_NAME")
    if notebook_name:
        candidates.append(Path.cwd() / str(notebook_name))
        project_root = globals().get("PROJECT_ROOT")
        if project_root:
            candidates.append(Path(project_root) / str(notebook_name))
    for candidate in candidates:
        try:
            if candidate.exists() and candidate.suffix.lower() == ".ipynb":
                return candidate
        except Exception:
            pass
    return None


_report_nb_path = _report_notebook_path()
_report_notebook = _report_nb_path.name if _report_nb_path is not None else str(globals().get("NOTEBOOK_NAME", "unknown_notebook"))
_report_category = globals().get("CATEGORY_ID", globals().get("CATEGORY_LABEL", "cross_category"))


def _report_path_from_maps(key):
    for map_name in ["OUTPUT_FILES", "OUTPUT_PATHS", "output_paths"]:
        mapping = globals().get(map_name)
        if isinstance(mapping, dict) and key in mapping:
            return str(mapping[key])
    return None


def _report_path_from_var(name):
    value = globals().get(name)
    return str(value) if value is not None else None


def _report_row_value(row, candidates):
    for column in candidates:
        if column in row and not _report_is_missing(row[column]):
            return row[column]
    return None


def _report_ci(row, low_candidates, high_candidates):
    lo = _report_row_value(row, low_candidates)
    hi = _report_row_value(row, high_candidates)
    if _report_is_missing(lo) or _report_is_missing(hi):
        return None
    return [_report_jsonable(lo), _report_jsonable(hi)]


def _report_value(claim_id, value=None, ci=None, p=None, n=None, source_file=None, aggregation=None, note=None):
    record = {
        "claim_id": str(claim_id),
        "value": _report_jsonable(value),
        "ci": _report_jsonable(ci),
        "p": _report_jsonable(p),
        "n": _report_jsonable(n),
        "source_file": _report_jsonable(source_file),
        "aggregation": _report_jsonable(aggregation),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_missing_value(claim_id, note="not_available_in_notebook"):
    return _report_value(
        claim_id=claim_id,
        value=None,
        ci=None,
        p=None,
        n=None,
        source_file=None,
        aggregation="not_available_in_notebook",
        note=note,
    )


def _report_gate(gate_id, observed=None, expected_contract="", self_flag=False, note=None):
    record = {
        "gate_id": str(gate_id),
        "observed": _report_jsonable(observed),
        "expected_contract": str(expected_contract),
        "self_flag": bool(self_flag),
    }
    if note:
        record["note"] = str(note)
    return record


def _report_frame_failed(frame, passed_columns=("passed", "check_passed", "identity_passed"), status_columns=("status", "coverage_status")):
    if not _report_is_dataframe(frame) or frame.empty:
        return False
    for column in passed_columns:
        if column in frame.columns:
            try:
                return not bool(frame[column].astype(bool).all())
            except Exception:
                pass
    for column in status_columns:
        if column in frame.columns:
            statuses = frame[column].astype(str).str.upper()
            return not bool(statuses.isin(["PASS", "SUCCESS", "TRUE"]).all())
    return False


def _report_filter_primary(frame):
    work = frame.copy()
    original = work
    if "metric_name" in work.columns:
        filtered = work.loc[work["metric_name"].astype(str).eq(str(globals().get("PRIMARY_METRIC_NAME", "NDCG")))]
        if not filtered.empty:
            work = filtered
    if "metric_cutoff" in work.columns:
        filtered = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(int(globals().get("PRIMARY_METRIC_CUTOFF", 5)))]
        if not filtered.empty:
            work = filtered
    if "candidate_pool_depth" in work.columns and "REPORT_POOL_DEPTH" in globals():
        filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["REPORT_POOL_DEPTH"]))]
        if not filtered.empty:
            work = filtered
    return work if not work.empty else original


def _report_values_15():
    values = []
    five = globals().get("stage_allocation_five_condition_comparison_df")
    if _report_is_dataframe(five):
        work = _report_filter_primary(five)
        value_columns = ["mean_metric_value", "metric_mean", "mean_value", "mean", "metric_value_mean", "mean_ndcg_at_5"]
        value_column = next((column for column in value_columns if column in work.columns), None)
        if value_column is None:
            values.append(_report_missing_value("section6_five_condition_means", "value_column_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [
                    row.get("stage_condition"), row.get("reranker_family"),
                    row.get("candidate_pool_depth"), row.get("metric_name"), row.get("metric_cutoff"),
                ]
                claim_id = "section6_condition_mean:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get(value_column),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low", "lower_ci"], ["bootstrap_ci_95_high", "ci_high", "upper_ci"]),
                    p=_report_row_value(row, ["p", "p_value", "sign_flip_p_value"]),
                    n=_report_row_value(row, ["case_count", "query_count", "n_cases", "n", "sample_size"]),
                    source_file=_report_path_from_maps("five_condition_comparison"),
                    aggregation=f"{value_column} from stage_allocation_five_condition_comparison_df",
                ))
    else:
        values.append(_report_missing_value("section6_five_condition_means"))

    contrasts = globals().get("stage_allocation_paired_contrasts_df")
    if _report_is_dataframe(contrasts):
        work = contrasts.copy()
        if "contrast_name" in work.columns:
            work = work.loc[work["contrast_name"].astype(str).eq("P2-P_minus_P2-Q")]
        if "metric_name" in work.columns:
            work = work.loc[work["metric_name"].astype(str).eq("NDCG")]
        if "metric_cutoff" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["metric_cutoff"], errors="coerce").eq(5)]
        if "candidate_pool_depth" in work.columns:
            work = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").isin([100, 300, 500, 700, 1000])]
        if "user_scope" in work.columns:
            work = work.loc[work["user_scope"].astype(str).eq("overall")]
        if "analysis_subset" in work.columns:
            work = work.loc[work["analysis_subset"].astype(str).eq("all_cases")]
        if work.empty:
            values.append(_report_missing_value("fig6_3_rankp_minus_base_ndcg5_by_depth", "requested_contrast_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                claim_id = "fig6_3_rankp_minus_base_ndcg5_by_depth:" + "|".join(str(row.get(column)) for column in ["reranker_family", "candidate_pool_depth"] if column in row)
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("mean_difference"),
                    ci=_report_ci(row, ["bootstrap_ci_95_low", "ci_low"], ["bootstrap_ci_95_high", "ci_high"]),
                    p=_report_row_value(row, ["sign_flip_p_value", "p_value", "p"]),
                    n=_report_row_value(row, ["paired_query_count", "n_pairs", "case_count", "n"]),
                    source_file=_report_path_from_maps("paired_contrasts"),
                    aggregation="existing paired mean_difference for P2-P_minus_P2-Q by depth",
                ))
    else:
        values.append(_report_missing_value("fig6_3_rankp_minus_base_ndcg5_by_depth"))
    return values


def _report_gates_15():
    gates = []
    manifest_obj = globals().get("manifest", {}) if isinstance(globals().get("manifest", {}), dict) else {}
    sig = globals().get("stage_allocation_significance_tests_df")
    observed_scope = {
        "analysis_role": manifest_obj.get("analysis_role"),
        "confirmatory_inference_authority_values": sorted(sig["confirmatory_inference_authority"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "confirmatory_inference_authority" in sig.columns else None,
        "inference_role_values": sorted(sig["inference_role"].dropna().astype(str).unique().tolist()) if _report_is_dataframe(sig) and "inference_role" in sig.columns else None,
    }
    gates.append(_report_gate("descriptive_only_scope", observed_scope, "descriptive-only; confirmatory inference authority remains Notebook 16", False))
    authority = globals().get("aggregate_metric_authority_qc_df")
    gates.append(_report_gate(
        "aggregate_metric_authority_qc",
        _report_frame_records(authority, limit=30),
        "descriptive aggregates reconcile with canonical authority",
        _report_frame_failed(authority, passed_columns=("check_passed", "passed")),
        None if _report_is_dataframe(authority) else "not_available_in_notebook",
    ))
    identity = globals().get("cold_fallback_identity_qc_df")
    gates.append(_report_gate(
        "cold_fallback_identity_qc",
        _report_frame_records(identity, limit=30),
        "fallback identity diagnostics are descriptive QC only",
        _report_frame_failed(identity),
        None if _report_is_dataframe(identity) else "not_available_in_notebook",
    ))
    gates.append(_report_gate(
        "stage_delta_fold_lineage_qc",
        manifest_obj.get("stage_delta_fold_lineage_qc", globals().get("stage_delta_fold_lineage_qc")),
        "stage-delta lineage recorded from existing Notebook 14/15 artifacts",
        False,
        None if (manifest_obj.get("stage_delta_fold_lineage_qc") is not None or "stage_delta_fold_lineage_qc" in globals()) else "not_available_in_notebook",
    ))
    return gates


def _report_values_18():
    values = []
    frame = globals().get("primary_feature_group_gain_summary_df")
    source_key = "primary_feature_group_gain_summary"
    if not _report_is_dataframe(frame):
        frame = globals().get("feature_group_gain_summary_df")
        source_key = "feature_group_gain_summary"
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "primary_report_depth" in work.columns:
            filtered = work.loc[work["primary_report_depth"].astype(bool)]
            if not filtered.empty:
                work = filtered
        elif "candidate_pool_depth" in work.columns and "PRIMARY_POOL_DEPTH" in globals():
            filtered = work.loc[_report_pd.to_numeric(work["candidate_pool_depth"], errors="coerce").eq(int(globals()["PRIMARY_POOL_DEPTH"]))]
            if not filtered.empty:
                work = filtered
        if "feature_group" not in work.columns or "mean_normalized_gain" not in work.columns:
            values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct", "required_columns_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"fig7_1_grouped_normalized_gain_pct:{row.get('feature_group')}",
                    value=None if _report_is_missing(row.get("mean_normalized_gain")) else float(row.get("mean_normalized_gain")) * 100.0,
                    ci=None,
                    p=None,
                    n=_report_row_value(row, ["fold_count", "n_folds", "n"]),
                    source_file=_report_path_from_maps(source_key),
                    aggregation="mean_normalized_gain across folds, expressed as percent",
                ))
    else:
        values.append(_report_missing_value("fig7_1_grouped_normalized_gain_pct"))
    return values


def _report_gates_18():
    gates = []
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    share_observed = {
        "manifest_flag": run.get("grouped_gain_shares_sum_to_one"),
        "primary_share_total": globals().get("_primary_share_total"),
        "group_share_totals": globals().get("_group_share_totals"),
    }
    share_self_flag = False
    if share_observed["manifest_flag"] is not None:
        share_self_flag = share_observed["manifest_flag"] is not True
    gates.append(_report_gate("grouped_gain_shares_sum_to_one", share_observed, "grouped gain shares sum to one within existing fold/depth summaries", share_self_flag))
    perm_flag = run.get("permutation_importance_computed")
    gates.append(_report_gate("permutation_importance_computed", perm_flag, "false in slim build manifest", perm_flag is not False, None if perm_flag is not None else "not_available_in_notebook"))
    shap_flag = run.get("shap_computed")
    gates.append(_report_gate("shap_computed", shap_flag, "false in slim build manifest", shap_flag is not False, None if shap_flag is not None else "not_available_in_notebook"))
    reproduction = globals().get("reproduction_qc_df")
    gates.append(_report_gate(
        "native_fold_prediction_reproduction",
        _report_frame_records(reproduction, limit=30),
        "native fold predictions reproduce strict OOF exports within recorded tolerance",
        _report_frame_failed(reproduction, passed_columns=("prediction_reproduced", "passed")),
        None if _report_is_dataframe(reproduction) else "not_available_in_notebook",
    ))
    return gates


def _report_values_19():
    values = []
    frame = globals().get("branch_contribution_df")
    required = list(globals().get("REQUIRED_BRANCH_GROUPS", ["candidate_common", "functional_prior", "brand_prior"]))
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "branch_group" in work.columns:
            filtered = work.loc[work["branch_group"].astype(str).isin(required)]
            if not filtered.empty:
                work = filtered
        if "importance_mean" not in work.columns:
            values.append(_report_missing_value("table7_1_branch_masking_delta", "importance_mean_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                parts = [row.get("branch_group"), row.get("user_scope"), row.get("regime")]
                claim_id = "table7_1_branch_masking_delta:" + "|".join(str(part) for part in parts if not _report_is_missing(part))
                values.append(_report_value(
                    claim_id=claim_id,
                    value=row.get("importance_mean"),
                    ci=_report_ci(row, ["importance_ci_95_low", "ci_low"], ["importance_ci_95_high", "ci_high"]),
                    p=None,
                    n=_report_row_value(row, ["case_count", "fold_count", "n"]),
                    source_file=_report_path_from_maps("branch_contribution"),
                    aggregation="mean NDCG@5 decrease from existing branch masking summary",
                ))
    else:
        values.append(_report_missing_value("table7_1_branch_masking_delta"))
    return values


def _report_gates_19():
    gates = []
    manifest_obj = globals().get("analysis_manifest", {}) if isinstance(globals().get("analysis_manifest", {}), dict) else {}
    checkpoint = globals().get("checkpoint_diagnostics_df")
    checkpoint_values = checkpoint["checkpoint_selection_metric"].dropna().astype(str).unique().tolist() if _report_is_dataframe(checkpoint) and "checkpoint_selection_metric" in checkpoint.columns else None
    observed_checkpoint = {
        "manifest_checkpoint_selection_metric": manifest_obj.get("checkpoint_selection_metric"),
        "checkpoint_diagnostics_values": checkpoint_values,
    }
    checkpoint_bad = manifest_obj.get("checkpoint_selection_metric") not in (None, "validation_ndcg_at_5")
    if checkpoint_values is not None:
        checkpoint_bad = checkpoint_bad or any(value != "validation_ndcg_at_5" for value in checkpoint_values)
    gates.append(_report_gate("checkpoint_selection_metric", observed_checkpoint, "validation_ndcg_at_5", checkpoint_bad))
    manifests = globals().get("interpretation_manifests", {}) if isinstance(globals().get("interpretation_manifests", {}), dict) else {}
    observed_mismatch = {
        "analysis_manifest_disabled_category_mismatch": manifest_obj.get("disabled_category_mismatch"),
        "full_manifest_disabled_category_mismatch": manifests.get("Full", {}).get("disabled_category_mismatch") if isinstance(manifests.get("Full", {}), dict) else None,
    }
    mismatch_bad = any(value is not None and value is not False for value in observed_mismatch.values())
    gates.append(_report_gate("disabled_category_mismatch", observed_mismatch, "false", mismatch_bad))
    return gates


def _report_values_20():
    values = []
    frame = globals().get("scorecard")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            claim_id = "table3_e1_structural_contrast:" + "|".join(str(row.get(column)) for column in ["claim_dimension", "metric"] if column in row)
            values.append(_report_value(
                claim_id=claim_id,
                value=row.get("value"),
                ci=None,
                p=None,
                n=_report_row_value(row, ["n", "case_count", "item_count", "query_count"]),
                source_file=str(_report_output_dir() / "schema_audit_thesis_evidence_scorecard.csv"),
                aggregation="existing schema-audit scorecard value",
            ))
    else:
        values.append(_report_missing_value("table3_e1_structural_contrast_values"))
    return values


def _report_gates_20():
    frame = globals().get("qc_summary")
    return [_report_gate(
        "schema_audit_qc_summary",
        _report_frame_records(frame, limit=50),
        "schema-audit QC rows reported by notebook",
        _report_frame_failed(frame, status_columns=("status",)),
        None if _report_is_dataframe(frame) else "not_available_in_notebook",
    )]


def _report_values_22():
    values = []
    frame = globals().get("paired_summary")
    requested = {
        "Actual_minus_Shuffled",
        "Actual_minus_QCHSfiltered",
        "Actual_minus_QCHS_filtered",
        "QCHS_minus_Actual",
        "Actual_minus_No_User_Brand",
    }
    if _report_is_dataframe(frame):
        work = frame.copy()
        if "population" in work.columns:
            work = work.loc[work["population"].astype(str).eq("non-cold")]
        if "contrast" in work.columns:
            work = work.loc[work["contrast"].astype(str).isin(requested)]
        if work.empty:
            values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas", "requested_non_cold_contrasts_not_available_in_notebook"))
        else:
            for row in work.to_dict(orient="records"):
                values.append(_report_value(
                    claim_id=f"table_a_2_prior_policy_delta:{row.get('contrast')}|non-cold",
                    value=row.get("mean_delta"),
                    ci=_report_ci(row, ["ci_low", "bootstrap_ci_95_low"], ["ci_high", "bootstrap_ci_95_high"]),
                    p=_report_row_value(row, ["p", "p_value"]),
                    n=_report_row_value(row, ["n_cases", "n", "case_count"]),
                    source_file=_report_path_from_var("PAIRED_SUMMARY_PATH"),
                    aggregation="existing non-cold paired_summary mean_delta and CI",
                ))
    else:
        values.append(_report_missing_value("table_a_2_prior_policy_non_cold_deltas"))
    return values


def _report_gates_22():
    gates = []
    paired = globals().get("paired_summary")
    if _report_is_dataframe(paired) and "population" in paired.columns:
        non_cold = paired.loc[paired["population"].astype(str).eq("non-cold")]
        observed_n = _report_frame_records(non_cold, limit=20, columns=["contrast", "population", "n_cases", "n_users"])
    else:
        observed_n = None
    gates.append(_report_gate("non_cold_n", observed_n, "non-cold n recorded in paired_summary", False, None if observed_n is not None else "not_available_in_notebook"))
    model_reuse = globals().get("model_reuse_decision")
    fold_qc = globals().get("fold_qc")
    reuse_observed = {
        "model_reuse_decision": _report_frame_records(model_reuse, limit=20),
        "fold_qc": _report_frame_records(fold_qc, limit=20),
        "canonical_fold_equivalence": globals().get("canonical_fold_equivalence"),
        "canonical_param_equivalence": globals().get("canonical_param_equivalence"),
        "canonical_preprocessing_equivalence": globals().get("canonical_preprocessing_equivalence"),
    }
    gates.append(_report_gate("nb11_fold_hyperparameter_reuse", reuse_observed, "NB11/P2-Q fold and hyperparameter reuse diagnostics are recorded", _report_frame_failed(model_reuse) or _report_frame_failed(fold_qc)))
    seed = globals().get("RANDOM_SEED")
    gates.append(_report_gate("seed", seed, "42", seed not in (None, 42), None if seed is not None else "not_available_in_notebook"))
    return gates


def _report_values_26():
    values = []
    frame = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            values.append(_report_value(
                claim_id=f"section7_1_strong_minus_weak:{row.get('reranker_family')}|{row.get('comparison_id')}",
                value=row.get("strong_minus_weak_delta"),
                ci=_report_ci(row, ["bootstrap_ci_95_lower", "ci_low"], ["bootstrap_ci_95_upper", "ci_high"]),
                p=None,
                n={"strong_case_count": row.get("strong_case_count"), "weak_case_count": row.get("weak_case_count")},
                source_file=_report_path_from_maps("strong_weak_difference_in_delta"),
                aggregation="existing independent strong-minus-weak bootstrap summary",
            ))
    else:
        values.append(_report_missing_value("section7_1_strong_minus_weak_delta_ci"))
    return values


def _report_gates_26():
    gates = []
    coverage = globals().get("regime_pair_coverage_qc")
    gates.append(_report_gate(
        "regime_pair_coverage",
        _report_frame_records(coverage, limit=80),
        "coverage_status PASS for each method x contrast x regime row",
        _report_frame_failed(coverage, status_columns=("coverage_status",)),
        None if _report_is_dataframe(coverage) else "not_available_in_notebook",
    ))
    strong_weak = globals().get("strong_weak_difference_in_delta")
    if _report_is_dataframe(strong_weak):
        observed_overlap = _report_frame_records(strong_weak, limit=30, columns=["reranker_family", "comparison_id", "strong_weak_case_overlap_count", "strong_weak_user_overlap_count"])
        self_flag = False
        for column in ["strong_weak_case_overlap_count", "strong_weak_user_overlap_count"]:
            if column in strong_weak.columns and _report_pd.to_numeric(strong_weak[column], errors="coerce").ne(0).any():
                self_flag = True
    else:
        observed_overlap = None
        self_flag = False
    gates.append(_report_gate("strong_weak_overlap_counts", observed_overlap, "case and user overlap counts equal 0", self_flag, None if observed_overlap is not None else "not_available_in_notebook"))
    return gates


def _report_values_28():
    values = []
    frame = globals().get("table_6_4")
    if _report_is_dataframe(frame):
        for row in frame.to_dict(orient="records"):
            key = "|".join(str(row.get(column)) for column in ["category_label", "reranker_family", "stage_condition", "candidate_pool_depth"] if column in row)
            values.append(_report_value(
                claim_id=f"table6_4_online_ms_per_query:{key}",
                value=row.get("online_ms_per_query"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="existing online_ms_per_query from table_6_4",
            ))
            values.append(_report_value(
                claim_id=f"table6_4_hardware:{key}",
                value=row.get("compute_device"),
                ci=None,
                p=None,
                n=None,
                source_file=_report_path_from_maps("table_6_4"),
                aggregation="hardware context from existing runtime table/source manifest",
            ))
    else:
        values.append(_report_missing_value("table6_4_method_depth_ms_per_query_and_hardware"))
    return values


def _report_gates_28():
    gates = []
    runtime_qc = globals().get("runtime_qc")
    four_checks = [
        "query_denominator_present",
        "hardware_fields_present",
        "pipeline_stage1_runtime_complete",
        "offline_components_not_in_online_latency",
    ]
    if _report_is_dataframe(runtime_qc) and "check" in runtime_qc.columns:
        raw_four = runtime_qc.loc[runtime_qc["check"].astype(str).isin(four_checks)].copy()
        if raw_four.empty:
            raw_four = runtime_qc.copy()
        observed_qc = _report_frame_records(raw_four, limit=20)
        self_flag = _report_frame_failed(raw_four)
    else:
        observed_qc = None
        self_flag = False
    gates.append(_report_gate("runtime_validation_qc", observed_qc, "four raw runtime validation QC items recorded", self_flag, None if observed_qc is not None else "not_available_in_notebook"))
    run = globals().get("run_manifest", {}) if isinstance(globals().get("run_manifest", {}), dict) else {}
    mode = run.get("mode")
    gates.append(_report_gate("aggregation_only", mode, "aggregation_only_no_retrain", mode not in (None, "aggregation_only_no_retrain"), None if mode is not None else "not_available_in_notebook"))
    separation = {
        "online_definition": run.get("online_definition"),
        "offline_excluded": run.get("offline_excluded"),
        "offline_gate": next((row for row in (observed_qc or []) if row.get("check") == "offline_components_not_in_online_latency"), None),
    }
    gates.append(_report_gate("online_offline_separation", separation, "online latency excludes offline components", False if separation["offline_gate"] is not None else True, None if separation["offline_gate"] is not None else "not_available_in_notebook"))
    return gates


def _report_kind():
    name = _report_notebook.lower()
    if name.startswith("15_") or "stage_allocation_summary" in name:
        return "15"
    if name.startswith("18_") or "lightgbm_heldout_interpretation" in name:
        return "18"
    if name.startswith("19_") or "transformer_heldout_interpretation" in name:
        return "19"
    if name.startswith("20_") or "category_schema_audit" in name:
        return "20"
    if name.startswith("22_") or "prior_policy_ablation_lightgbm" in name:
        return "22"
    if name.startswith("26_") or "regime_effect_summary" in name:
        return "26"
    if name.startswith("28_") or "runtime_efficiency_summary" in name:
        return "28"
    return "unknown"


_REPORT_KIND = _report_kind()
_REPORT_SERVED = {
    "15": ["section6_condition_means", "fig6_3"],
    "18": ["fig7_1"],
    "19": ["table7_1", "section7_3"],
    "20": ["table3_e1", "appendix3_e"],
    "22": ["section7_2", "table_a_2"],
    "26": ["section7_1"],
    "28": ["table6_4"],
}.get(_REPORT_KIND, [])
_REPORT_VALUE_BUILDERS = {
    "15": _report_values_15,
    "18": _report_values_18,
    "19": _report_values_19,
    "20": _report_values_20,
    "22": _report_values_22,
    "26": _report_values_26,
    "28": _report_values_28,
}
_REPORT_GATE_BUILDERS = {
    "15": _report_gates_15,
    "18": _report_gates_18,
    "19": _report_gates_19,
    "20": _report_gates_20,
    "22": _report_gates_22,
    "26": _report_gates_26,
    "28": _report_gates_28,
}


def _report_collect_inputs():
    records = []
    seen = set()

    def add_path(path, sha=None):
        if _report_is_missing(path):
            return
        text = str(path).strip()
        if not text:
            return
        candidate = Path(text)
        if _report_is_inside(candidate, _report_output_dir()):
            return
        key = str(candidate)
        if key in seen:
            return
        seen.add(key)
        records.append({"path": key, "sha256": _report_jsonable(sha if sha is not None else _report_file_sha256(candidate, allow_heavy=False))})

    def looks_like_path(text):
        suffix = Path(str(text)).suffix.lower()
        return suffix in {".json", ".csv", ".parquet", ".txt", ".yaml", ".yml", ".ipynb", ".pkl", ".pickle", ".joblib", ".pt", ".pth", ".bin"}

    def walk(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                key_text = str(key).lower()
                if isinstance(value, (str, Path)) and (key_text.endswith("path") or key_text.endswith("paths") or looks_like_path(value)):
                    sha = None
                    if key_text.endswith("path"):
                        sha = obj.get(str(key).replace("path", "sha256")) or obj.get(str(key).replace("_path", "_sha256"))
                    add_path(value, sha)
                else:
                    walk(value)
        elif isinstance(obj, (list, tuple, set)):
            for value in obj:
                walk(value)
        elif isinstance(obj, (str, Path)) and looks_like_path(obj):
            add_path(obj)

    for obj_name in ["pipeline_manifest", "run_manifest", "analysis_manifest", "manifest"]:
        obj = globals().get(obj_name)
        if isinstance(obj, dict):
            walk(obj)

    inventory = globals().get("runtime_inventory")
    if _report_is_dataframe(inventory):
        for _, row in inventory.iterrows():
            for path_col in ["source_path", "source_manifest", "manifest_path", "path"]:
                if path_col in inventory.columns:
                    sha = None
                    for sha_col in [path_col.replace("path", "sha256"), "source_sha256", "sha256"]:
                        if sha_col in inventory.columns:
                            sha = row.get(sha_col)
                            break
                    add_path(row.get(path_col), sha)

    for name, value in list(globals().items()):
        if not name.endswith("_PATH"):
            continue
        if name.startswith(("OUTPUT", "PREDICTIONS", "PER_CASE", "DELTAS", "PAIRED_SUMMARY", "POPULATION_RESULTS", "PROFILE_DIAGNOSTICS", "SHUFFLE_ASSIGNMENT", "FALLBACK_QC", "FEATURE_CONTRACT", "MODEL_REUSE", "FOLD_QC", "LEAKAGE_QC", "COVERAGE", "RETENTION", "FEATURE_IMPORTANCE", "QC_SUMMARY", "MANIFEST")):
            if _report_is_inside(value, _report_output_dir()):
                continue
        add_path(value)

    return records


_report_run_utc = datetime.now(timezone.utc).isoformat()
_report_values = _REPORT_VALUE_BUILDERS.get(_REPORT_KIND, lambda: [_report_missing_value("requested_values")])()
_report_gates = _REPORT_GATE_BUILDERS.get(_REPORT_KIND, lambda: [_report_gate("requested_gates", None, "not_available_in_notebook", False, "not_available_in_notebook")])()
_report_lineage = {
    "notebook": _report_notebook,
    "category": _report_jsonable(_report_category),
    "run_utc": _report_run_utc,
    "code_sha": _report_file_sha256(_report_nb_path, allow_heavy=True) if _report_nb_path is not None else None,
    "inputs": _report_collect_inputs(),
}


def _report_md_cell(value):
    text = "" if value is None else (_report_jsonable(value))
    if isinstance(text, (dict, list)):
        text = json.dumps(text, ensure_ascii=False, sort_keys=True)
    text = str(text)
    return text.replace("|", "\\|").replace("\n", "<br>")


def _report_markdown(values, gates, lineage):
    lines = []
    lines.append("# Identity & lineage")
    lines.append(f"- notebook: {_report_md_cell(lineage.get('notebook'))}")
    lines.append(f"- category: {_report_md_cell(lineage.get('category'))}")
    lines.append(f"- run_utc: {_report_md_cell(lineage.get('run_utc'))}")
    lines.append(f"- code_sha: {_report_md_cell(lineage.get('code_sha'))}")
    lines.append(f"- input_count: {len(lineage.get('inputs', []))}")
    lines.append("")
    lines.append("# Served thesis elements")
    if _REPORT_SERVED:
        for served_id in _REPORT_SERVED:
            lines.append(f"- {_report_md_cell(served_id)}")
    else:
        lines.append("- not_available_in_notebook")
    lines.append("")
    lines.append("# Computed headline values")
    value_columns = ["claim_id", "value", "ci", "p", "n", "source_file", "aggregation", "thesis_value"]
    lines.append("| " + " | ".join(value_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(value_columns)) + " |")
    for record in values:
        row = dict(record)
        row["thesis_value"] = ""
        lines.append("| " + " | ".join(_report_md_cell(row.get(column)) for column in value_columns) + " |")
    lines.append("")
    lines.append("# QC gates")
    gate_columns = ["gate_id", "observed", "expected_contract", "self_flag"]
    lines.append("| " + " | ".join(gate_columns) + " |")
    lines.append("| " + " | ".join(["---"] * len(gate_columns)) + " |")
    for record in gates:
        lines.append("| " + " | ".join(_report_md_cell(record.get(column)) for column in gate_columns) + " |")
    lines.append("")
    lines.append("# Self-detected anomalies")
    anomalies = []
    for record in gates:
        if record.get("self_flag") is True:
            anomalies.append(f"- gate_self_flag: {_report_md_cell(record.get('gate_id'))}")
        if record.get("note"):
            anomalies.append(f"- gate_note: {_report_md_cell(record.get('gate_id'))}: {_report_md_cell(record.get('note'))}")
    for record in values:
        if record.get("note"):
            anomalies.append(f"- value_note: {_report_md_cell(record.get('claim_id'))}: {_report_md_cell(record.get('note'))}")
    lines.extend(anomalies if anomalies else ["- none"])
    lines.append("")
    return "\n".join(lines)


_lineage_path = _report_dir / "lineage.json"
_values_path = _report_dir / "report_values.json"
_gates_path = _report_dir / "qc_gates.json"
_markdown_path = _report_dir / "verification_report.md"
_lineage_path.write_text(json.dumps(_report_lineage, ensure_ascii=False, indent=2), encoding="utf-8")
_values_path.write_text(json.dumps(_report_values, ensure_ascii=False, indent=2), encoding="utf-8")
_gates_path.write_text(json.dumps(_report_gates, ensure_ascii=False, indent=2), encoding="utf-8")
_markdown_path.write_text(_report_markdown(_report_values, _report_gates, _report_lineage), encoding="utf-8")
for _written_path in [_lineage_path, _values_path, _gates_path, _markdown_path]:
    print(_written_path)


/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/regime_analysis_report/report/lineage.json
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/regime_analysis_report/report/report_values.json
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/regime_analysis_report/report/qc_gates.json
/content/drive/MyDrive/thesis_recsys/categories/facial_skincare/outputs/analysis/regime_analysis_report/report/verification_report.md
